<a href="https://colab.research.google.com/github/Geauga/Proteina-Complexa-dev/blob/main/coLab/Proteina_Complexa_coLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  # Section 0: User Quick Start README & Global Configuration (State Saved; Re-run Optional After Restart)

<details>
<summary><font size="5"><b>⚙️ Expand: User Quick Start README & Global Configuration</b></font></summary>

<br>

### 💡 ARCHITECTURE OVERVIEW
This notebook employs a decoupled "Front-end UI / Back-end Engine" architecture. Please read this quick start guide and configure all parameters exclusively within the **Section 0** dashboard. This centralized panel persists your configuration data to Google Drive, ensuring seamless execution even across kernel restarts.

### 🛑 CRITICAL NOTICE: MANDATORY KERNEL RESTART
After completing **Section 1** (Environment Configuration), a Colab runtime restart is **strictly required** to apply core library upgrades (JAX/Flax/CUDA).
**ACTION:** Navigate to `Runtime -> Restart session` from the menu, and then **run Section 1 again**. Failure to do so will cause the pipeline to terminate with an error.

### 📦 ZERO-CONFIGURATION ASSETS
Demonstration protein structures (`GFP_1_10.pdb` and `GFP_11.pdb`) are pre-integrated.
* **Custom Targets:** Upload your PDB files to the `Proteina-Complexa/targets/` directory on Google Drive, or use the integrated upload toggle in Section 0.
* **Configuration:** After uploading, ensure you update the `task_name` and `pdb_file_name` in Section 0 accordingly.

---
### 🔄 EXECUTION WORKFLOWS

**Phase I: Universal Initialization**
1.  **Hardware Requirements & Pre-check:**
    * **Storage:** At least 20GB of free space on Google Drive is recommended; 30GB+ is required if localizing program initialization data to accelerate startup.
    * **GPU:** A minimum of L4 GPU runtime is required. For longer protein chains with high VRAM usage, A100 runtime is needed. *Note: The GFP tutorial can be completed using L4 runtime with default settings.*
2.  **Read & Configure:** Review these guidelines and adjust all parameters (Auto-Pilot, Standard, Geometric Scanning) in the **Section 0** UI dashboard. Execute the cell to register the variables persistently.
3.  **Environment Initialization:** Execute **Section 1**.
    * *Note:* Initial compilation and asset synchronization take approximately 10–20 minutes. Enabling Google Drive persistent backup (~10GB) is recommended, allowing subsequent runs to skip this stage even if the session is reset.
4.  **Restart Session & Verification:** Perform `Runtime -> Restart session`. **You must re-run Section 1 after the restart** (re-running Section 0 is optional).
5.  **Target Pre-processing:** Expand and execute **Section 2** to parse and prepare the target structure.

**Phase II: Divergent Execution Pathways**
* **Section 3: GFP Tutorial.** The system automatically scans the surface of Split-GFP subunit 1 to identify the subunit 2 binding groove. It then calls the `Proteina_Complexa` evaluation module to generate the complex, providing baseline data for the natural peptide and 3D structural models of the complex.
* **Section 4: High-Throughput Auto-Pilot.** Continuously generates, evaluates, and logs candidates until a predefined **success threshold** is met. Default is set to screen for subunit 1 binding peptides; users can upload or specify their target protein in the Section 0 panel.
* **Section 5: Targeted Surface Screening.** Screens for the hotspots with the strongest binding affinity on the target protein's surface and their corresponding peptides. Default is set to screen for subunit 1 binding peptides; users can upload or specify their target protein in the Section 0 panel.

</details>

In [16]:
# @title 🎯 Section_0_Global_Configuration_Dashboard { display-mode: "form" }
# @markdown Complete this dashboard **ONCE**. All settings are persistently saved to your Google Drive.

import os, json, shutil, sys
from pathlib import Path
from IPython.display import display, HTML
import py3Dmol

# ==========================================
# 0. Infrastructure & Google Drive Mount
# ==========================================
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    print("🔄 Mounting Google Drive for persistent storage...")
    drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

# --- Source Repository Detection and Cloning ---
if not os.path.exists('/content/drive/MyDrive/Proteina-Complexa/pyproject.toml'):
    print(f">>> Source repository not detected. Executing initial clone to {ROOT_DIR}...")
    !git clone https://github.com/Geauga/Proteina-Complexa-coLab {ROOT_DIR}
else:
    print(f">>> Source repository already exists at {ROOT_DIR}. Skipping clone step.")

TARGETS_DIR = BASE_DIR / 'targets'
TARGETS_DIR.mkdir(parents=True, exist_ok=True)
SETTING_FILE = BASE_DIR / 'internal_settings.json'
# @markdown ---
# @markdown ---
# ==========================================
# --- Configuration for Environment Setup & Cache (Cell 3) ---
# ==========================================
# @markdown ### 💾 Environment & Cache Settings
# @markdown **[Accelerate Startup]** Enable Google Drive backup for the build cache?
# @markdown > *Note: The very first run will be slightly slower (1-2 mins) to package the backup; subsequent startups will be lightning fast.*
enable_cache_backup = True # @param {type:"boolean"}

# ==========================================
# --- Configuration for Target Setup & Processing (Cell 4) ---
# ==========================================
# @markdown ---
# @markdown ---
# @markdown ### 📁 Target File Selection
UPLOAD_NEW_PDB = False #@param {type:"boolean"}
pdb_file_name = "GFP_1_10.pdb" #@param {type:"string"}

# @markdown ### 🧬 Target Definition
task_name = "GFP" #@param {type:"string"}
target_chains = "A" #@param {type:"string"}
# @markdown ---
# @markdown ---
run_filter = True #@param {type:"boolean"}
run_evaluate = True #@param {type:"boolean"}
run_analyze = True #@param {type:"boolean"}

# @markdown ---
# @markdown ---
# @markdown ### 🎯 Interaction Hotspots (Comma separated)
hotspots_input = "39,40,41,73,74,75,75,77,78,199,200,201,202,203,204" #@param {type:"string"}

# @markdown ### 📏 Binder Peptide Length
binder_length_min = 10 #@param {type:"integer"}
binder_length_max = 15 #@param {type:"integer"}
# @markdown ---
# @markdown ---
# @markdown ### 🎯 Auto-Pilot Screening Configuration (Section 4)
target_threshold = 0.85 #@param {type:"number"}
target_success_count = 10 #@param {type:"integer"}
max_iterations = 5 #@param {type:"integer"}

batch_size = 36 #@param {type:"integer"}
designs_per_loop = 36 #@param {type:"integer"}



# ==========================================
# --- Geometric Surface Scanning Config (Section 6) ---
# ==========================================
# @markdown ---
# @markdown ---
# @markdown ### 📐 Geometric Surface Scanning
anchor_residues = "39,40,41,73,74,75,75,77,78,199,200,201,202,203,204" #@param {type:"string"}
scan_margin = 10.0 #@param {type:"number"}
local_patch_radius = 8.0 #@param {type:"number"}
scan_batch_size = 36 #@param {type:"integer"}
designs_per_patch = 36 #@param {type:"integer"}
# ==========================================
# --- Automated Multi-Dimensional Surface Scanning (Cell 11) ---
# ==========================================
# @markdown ---
# @markdown ---
# @markdown ### 🔬 Automated Groove & Pocket Discovery
scan_shallow_groove = True #@param {type:"boolean"}
scan_deep_pocket = True #@param {type:"boolean"}
scan_hydrophobic_patch = True #@param {type:"boolean"}
scan_charged_patch = True #@param {type:"boolean"}



# ==========================================
# Execution Logic: Validation & Persistence
# ==========================================

final_pdb_name = pdb_file_name

if UPLOAD_NEW_PDB:
    from google.colab import files
    display(HTML('<p style="font-size:16px; font-weight:bold; color:#E32636;">👉 PLEASE CLICK "CHOOSE FILES" TO UPLOAD:</p>'))
    uploaded = files.upload()
    if uploaded:
        for filename in uploaded.keys():
            shutil.move(filename, TARGETS_DIR / filename)
            final_pdb_name = filename
            break

# Bundle settings
settings = {
    "enable_cache_backup": enable_cache_backup,
    "pdb_file": final_pdb_name,
    "task_name": task_name,
    "target_chains": target_chains,
    "hotspots_input": hotspots_input,
    "binder_length_min": binder_length_min,
    "binder_length_max": binder_length_max,

    "run_filter": run_filter,
    "run_evaluate": run_evaluate,
    "run_analyze": run_analyze,
    "target_threshold": target_threshold,
    "target_success_count": target_success_count,
    "designs_per_loop": designs_per_loop,
    "max_iterations": max_iterations,
    "batch_size": batch_size,
    "anchor_residues": anchor_residues,
    "scan_margin": scan_margin,
    "local_patch_radius": local_patch_radius,
    "designs_per_patch": designs_per_patch,

    "scan_batch_size": scan_batch_size,

}

# Persistent Save
with open(SETTING_FILE, 'w') as f:
    json.dump(settings, f, indent=4)

print(f"\n✅ SETTINGS REGISTERED")
print(f"Task Name  : {task_name}")
print(f"Target PDB : {final_pdb_name} (Chain: {target_chains})")

print("💾 Data saved. The backend engines will seamlessly inherit these unified configurations.")

>>> Source repository already exists at /content/drive/MyDrive/Proteina-Complexa. Skipping clone step.

✅ SETTINGS REGISTERED
Task Name  : GFP
Target PDB : GFP_1_10.pdb (Chain: A)
💾 Data saved. The backend engines will seamlessly inherit these unified configurations.


# Section 1: Environment Configuration & Dependency Management (Run All 4 Cells, Then Restart Session)

In [22]:
# Cell_1_Environment_Initialization.py
# Requirement: Upgrade essential logging and serialization dependencies (wandb, protobuf) for the pipeline.

# ==========================================
# --- Environment Base Dependencies ---
# ==========================================
!pip install --upgrade wandb protobuf

# Purpose: Ensure the base environment has the latest required versions of Weights & Biases (wandb) for experiment tracking and protocol buffers (protobuf) for model serialization.
# Upstream Code: The first cell of the provided Colab notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 08:30 EDT.
# Changed Lines:
# * Designated the cell as "Cell 1" in the header.
# * Appended the standard metadata tracking block.

  Using cached wandb-0.25.1-py3-none-manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached wandb-0.25.1-py3-none-manylinux_2_28_x86_64.whl (25.6 MB)
  Attempting uninstall: wandb
    Found existing installation: wandb 0.23.1
    Uninstalling wandb-0.23.1:
      Successfully uninstalled wandb-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
proteinfoundation 1.0.0 requires scipy==1.12.0, but you have scipy 1.17.1 which is incompatible.
proteinfoundation 1.0.0 requires wandb==0.23.1, but you have wandb 0.25.1 which is incompatible.


In [23]:
# Cell_2_Mount_Google_Drive.py
# Requirement: Mount Google Drive to the Colab instance to enable access to persistent storage for the Proteina-Complexa project files.
import py3Dmol
from google.colab import drive
drive.mount('/content/drive')

# Purpose: Connect the Google Colab virtual machine to the user's Google Drive, allowing the pipeline to read configurations, load models, and save generated protein structures persistently.
# Upstream Code: The original Google Drive mounting cell (formerly labeled as Cell 1) from the uploaded notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 08:35 EDT.
# Changed Lines:
# * Designated the cell as "Cell 2" in the header.
# * Appended the standard metadata tracking block.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Cell_2b_environment_setup_engine.py
import os,json
from pathlib import Path
from IPython.display import display, HTML

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

SETTING_FILE = BASE_DIR / 'internal_settings.json'
# --- Load Persistent Variables ---
enable_cache_backup = True
if os.path.exists(SETTING_FILE):
    with open(SETTING_FILE, 'r') as f:
        settings = json.load(f)
        enable_cache_backup = settings.get("enable_cache_backup", True)
else:
    print(f">>> Warning: {SETTING_FILE} not found. Defaulting enable_cache_backup to True.")

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

%cd {ROOT_DIR}

def setup_environment():
    print(">>> Initializing installation and environment configuration...")

    # 1. Base Installation
    !pip install uv
    !uv pip install --system --index-strategy unsafe-best-match -e .

    # 2. Consolidated Dependency Installation
    !uv pip install --system git+https://github.com/RosettaCommons/atomworks.git
    !uv pip install --system torch-scatter --no-build-isolation
    !uv pip install --system dm-haiku openbabel-wheel graphein e3nn

    # 3. Deploy Foldseek Binary
    if not os.path.exists(".venv/bin/foldseek"):
        !mkdir -p .venv/bin
        !wget -q -nc https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
        !tar xzf foldseek-linux-avx2.tar.gz
        !mv foldseek/bin/foldseek .venv/bin/ && chmod +x .venv/bin/foldseek
        !rm -rf foldseek-linux-avx2.tar.gz foldseek/

    # 4. Create sc Script Mock
    !mkdir -p env/docker/internal/
    !echo -e '#!/bin/bash\necho "0.000"' > env/docker/internal/sc && chmod +x env/docker/internal/sc

    # 5. Initialization and Model Download
    !complexa init uv
    !source env.sh && complexa download --complexa-all
    !source env.sh && complexa download --all

    # 6. Path Correction (.env)
    code_path = ROOT_DIR + "/"
    data_path = os.path.join(ROOT_DIR, "assets/")
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE, "r") as f: lines = f.readlines()
        with open(ENV_FILE, "w") as f:
            for line in lines:
                if line.startswith("LOCAL_CODE_PATH="): f.write(f"LOCAL_CODE_PATH={code_path}\n")
                elif line.startswith("LOCAL_DATA_PATH="): f.write(f"LOCAL_DATA_PATH={data_path}\n")
                else: f.write(line)

    # 7. Task Completion: Conditional Backup (Reads enable_cache_backup from Cell 2a)
    if enable_cache_backup:
        if not os.path.exists(UV_CACHE_TAR):
            print(">>> User opted in for cache backup. Extracting and backing up underlying C++ build cache to Drive...")
            !mkdir -p {BACKUP_DIR}
            !mkdir -p ~/.cache/uv
            !cd ~ && tar -cf {UV_CACHE_TAR} .cache/uv
            print(">>> Cache backup complete. Future reboots will achieve rapid recovery.")
        else:
            print(">>> Existing cache backup detected in Drive. Skipping redundant packaging.")
    else:
        print(">>> Cache backup bypassed per user configuration.")

# ====================================================================
# --- Main Logic: Inject cache first, then execute configuration ---
# ====================================================================
if os.path.exists(UV_CACHE_TAR):
    print(">>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...")
    !mkdir -p ~/.cache
    !tar -xf {UV_CACHE_TAR} -C ~
    print(">>> Cache injection complete. Preparing environment reconstruction...")
    setup_environment()
else:
    print(">>> No cache detected. Commencing initial heavy compilation (please wait)...")
    setup_environment()

print(">>> Environment ready. Initiating final validation...")
!source env.sh && complexa validate design configs/search_binder_local_pipeline.yaml

/content/drive/MyDrive/Proteina-Complexa
>>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...
>>> Cache injection complete. Preparing environment reconstruction...
>>> Initializing installation and environment configuration...
Using Python 3.12.13 environment at: /usr
Resolved 138 packages in 6.25s
Prepared 1 package in 2.50s
Uninstalled 5 packages in 173ms
Installed 5 packages in 26ms
 - biotite==1.4.0
 + biotite==0.41.2
 - numpy==2.4.4
 + numpy==1.26.4
 ~ proteinfoundation==1.0.0 (from file:///content/drive/MyDrive/Proteina-Complexa)
 - scipy==1.17.1
 + scipy==1.12.0
 - wandb==0.25.1
 + wandb==0.23.1
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 4.32s
Uninstalled 2 packages in 25ms
Installed 2 packages in 29ms
 - biotite==0.41.2
 + biotite==1.4.0
 - scipy==1.12.0
 + scipy==1.17.1
Using Python 3.12.13 environment at: /usr
Checked 1 package in 99ms
Using Python 3.12.13 environment at: /usr
Checked 4 packages in 107ms

  env.sh already exists.

In [25]:
# Cell_3_Repository_Clone_and_JAX_Patch.py
# Requirement: Detect and clone the Proteina-Complexa repository if absent. Upgrade core machine learning libraries (JAX, Flax, Haiku), and dynamically patch deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.

# ==========================================
# --- Repository Initialization & Environment Patching ---
# ==========================================
import os
import sys
import subprocess
from pathlib import Path

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")


# Align Paths
BASE_DIR = Path(ROOT_DIR)
COLABDESIGN_DIR = BASE_DIR / 'community_models/colabdesign'

def run_command(command_list):
    print(f"🚀 Executing: {' '.join(command_list)[:80]}...")
    process = subprocess.Popen(command_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout: print(line, end='', flush=True)
    process.wait()

# 1. Upgrade core dependencies (tailored for GPU environment)
print("\n[Stage 1/3] Upgrading JAX and core ecosystem libraries...")
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "jax[cuda12]", "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"])
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "flax", "dm-haiku", "chex", "optax"])

# 2. Patch JAX underlying NumPy compatibility (for Python 3.12+)
print("\n[Stage 2/3] Patching JAX and NumPy conflicts...")
literals_path = Path('/usr/local/lib/python3.12/dist-packages/jax/_src/literals.py')
if literals_path.exists():
    code = literals_path.read_text()
    if ", copy=copy" in code:
        literals_path.write_text(code.replace(", copy=copy", ""))
        print("✅ JAX copy=copy patch applied successfully!")

# 3. Deep patching for ColabDesign source code
print("\n[Stage 3/3] Patching ColabDesign source code compatibility...")
if COLABDESIGN_DIR.exists():
    for filepath in COLABDESIGN_DIR.rglob('*.py'):
        code = filepath.read_text()
        new_code = code.replace('jax.lib.xla_bridge.get_backend()', 'jax.extend.backend.get_backend()') \
                       .replace('.live_buffers()', '.live_arrays()') \
                       .replace('jax.tree_map', 'jax.tree_util.tree_map') \
                       .replace('jax.tree_flatten', 'jax.tree_util.tree_flatten') \
                       .replace('jax.tree_unflatten', 'jax.tree_util.tree_unflatten') \
                       .replace('jax.tree_leaves', 'jax.tree_util.tree_leaves')
        if '@jax.util.wraps' in new_code:
            new_code = new_code.replace('@jax.util.wraps(fun, docstr=docstr)', '@functools.wraps(fun)') \
                               .replace('@jax.util.wraps(fun)', '@functools.wraps(fun)')
            if 'import functools' not in new_code: new_code = 'import functools\n' + new_code
        if new_code != code: filepath.write_text(new_code)
    print("✅ ColabDesign patching completed.")
else:
    print("⚠️ ColabDesign directory not found. Skipping Stage 3 patching.")

print("\n✨ Environment patching is complete. You may now run the evaluation cells below.")

# Purpose: Integrate repository cloning with machine learning dependency upgrades. Patches deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.
# Upstream Code: User provided combined logic for Git cloning and JAX/ColabDesign patching.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:42 EDT.
# Changed Lines:
# * Consolidated duplicate imports and harmonized base directory variables natively.
# * Added an `else` safeguard block for Stage 3 in case the ColabDesign directory is absent at runtime.


[Stage 1/3] Upgrading JAX and core ecosystem libraries...
🚀 Executing: /usr/bin/python3 -m pip install --upgrade jax[cuda12] -f https://storage.googlea...
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
proteinfoundation 1.0.0 requires scipy==1.12.0, but you have scipy 1.17.1 which is incompatible.
graphein 1.7.8 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have

# Section 2: Target PDB Parsing & Structural Pre-processing

In [ ]:

# Cell_4b_PDB_Processing_and_YAML_Configuration_Generation.py
# Requirement: Process user PDB and dynamically build the target configuration yaml. Scan the 'coLab' directory and copy all .pdb files to the 'targets' directory, skipping existing files.

import os, json, yaml, shutil, time, sys
from pathlib import Path

print("🔍 Initializing Target Processing Engine...")

# ---------------------------------------------------------
# 1. Load Persistent Configuration from Google Drive
# ---------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

SETTING_FILE = Path('/content/drive/MyDrive/Proteina-Complexa/internal_settings.json')
if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found.")
    print("💡 Action: Please scroll up to Section 0, complete the form, and run the Global Dashboard cell first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

# Extract essential variables dynamically
pdb_file = cfg['pdb_file']
task_name = cfg['task_name']
target_chains = cfg['target_chains']
hotspots_input = str(cfg['hotspots_input'])
binder_length_min = int(cfg['binder_length_min'])
binder_length_max = int(cfg['binder_length_max'])

print(f"✅ Loaded persistent settings for task: [{task_name}]")

# ---------------------------------------------------------
# 2. Path Setup & Verification
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGETS_INPUT_DIR = BASE_DIR / 'targets'
RUN_DATA_DIR = BASE_DIR / 'assets' / 'target_data' / task_name
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'
DEMO_DIR = BASE_DIR / 'coLab'

TARGETS_INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- [Logic Add: Batch Demo File Auto-Initialization] ---
print("📦 Synchronizing demo PDB files from repository...")
if DEMO_DIR.exists():
    for pdb_item in DEMO_DIR.glob('*.pdb'):
        dest_item = TARGETS_INPUT_DIR / pdb_item.name
        if not dest_item.exists():
            shutil.copy(pdb_item, dest_item)
            print(f"  -> Copied: {pdb_item.name}")
        else:
            print(f"  -> Skipped (Already exists): {pdb_item.name}")
else:
    print(f"⚠️ Warning: Source directory '{DEMO_DIR.name}' not found.")

raw_pdb_path = TARGETS_INPUT_DIR / pdb_file
fixed_pdb_path = RUN_DATA_DIR / f"{task_name}_fixed.pdb"

if not raw_pdb_path.exists():
    print(f"❌ Error: Required PDB file '{pdb_file}' not found in '{TARGETS_INPUT_DIR}'.")
    sys.exit(1)

# ---------------------------------------------------------
# 3. PDB Parsing & Missing Gap Stitching
# ---------------------------------------------------------
print(f"⚙️ Parsing raw structure '{pdb_file}' and stitching missing residues...")

valid_res_nums = set()
with open(raw_pdb_path, 'r') as f:
    for line in f:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA' and line[21] == target_chains:
            try: valid_res_nums.add(int(line[22:26].strip()))
            except: continue

with open(raw_pdb_path, 'r') as f_in, open(fixed_pdb_path, 'w') as f_out:
    for line in f_in:
        if line.startswith('ATOM') and line[21] == target_chains:
            try:
                if int(line[22:26].strip()) in valid_res_nums: f_out.write(line)
            except: pass
        elif line.startswith('TER'): f_out.write(line)

res_nums = sorted(list(valid_res_nums))
ranges = []
if res_nums:
    start = prev = res_nums[0]
    for n in res_nums[1:]:
        if n == prev + 1: prev = n
        else:
            ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")
            start = prev = n
    ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")

# ---------------------------------------------------------
# 4. YAML Configuration Generation
# ---------------------------------------------------------
hotspots_list = [int(x.strip()) for x in hotspots_input.split(',') if x.strip().isdigit()]

yaml_data = {'target_dict_cfg': {
    task_name: {
        'target_input': ','.join(ranges),
        'binder_length': [binder_length_min, binder_length_max],
        'target_chains': [target_chains],
        'pdb_id': task_name,
        'source': task_name,
        'target_filename': task_name + "_fixed",
        'hotspot_residues': [f"{target_chains}{i}" for i in hotspots_list]
    }
}}

YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
if YAML_PATH.exists():
    backup_name = f"targets_dict_backup_{int(time.time())}.yaml"
    shutil.copy(YAML_PATH, YAML_PATH.parent / backup_name)
    print(f"💾 Existing config safely backed up as: {backup_name}")

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"🎯 SUCCESS: Target configuration generated dynamically for '{task_name}'. Ready for generation phase!")

# Purpose: Process PDB, build YAML configuration, and batch copy all demo PDB files from the repository clone to the working targets directory without overwriting existing files.
# Upstream Code: Configuration read from Section 0 (Global Dashboard).
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 14:53 EDT.
# Changed Lines:
# * Line 38: Replaced single DEMO_PDB_SOURCE file path with DEMO_DIR directory path.
# * Lines 44-53: Replaced the single file copy logic with a for-loop iterating over DEMO_DIR.glob('*.pdb'), checking dest_item.exists() before executing shutil.copy().

🔍 Initializing Target Processing Engine...
✅ Loaded persistent settings for task: [GFP]
📦 Synchronizing demo PDB files from repository...
  -> Skipped (Already exists): GFP_11.pdb
  -> Skipped (Already exists): GFP_1_10.pdb
  -> Skipped (Already exists): PD-L1.pdb
⚙️ Parsing raw structure 'GFP_1_10.pdb' and stitching missing residues...
💾 Existing config safely backed up as: targets_dict_backup_1775555573.yaml
🎯 SUCCESS: Target configuration generated dynamically for 'GFP'. Ready for generation phase!


# Section 3: Split-GFP Tutorial — Natural Baseline Evaluation & Complex Generation

In [ ]:
# Cell_5_Automated_Groove_Scanning_Engine.py
# Requirement: Implement Morphological Bottleneck Severing via EDT to mathematically sever narrow topological channels.
# Integrate in-notebook py3Dmol visualization to render the host protein as a gray mesh and highlight identified grooves as an orange surface.

import os, json, sys, yaml
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import KDTree
from scipy.ndimage import binary_closing, distance_transform_edt, label

from IPython.display import display, HTML

print("🧬 Initializing Morphological Bottleneck Severing Engine (Optimized for Split-Barrels)...")

# ---------------------------------------------------------
# 1. Load Configurations & YAML Interface
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

if not SETTING_FILE.exists() or not YAML_PATH.exists():
    print("❌ Error: Required configuration files not found.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_chains = cfg['target_chains']

scan_shallow_groove = bool(cfg.get('scan_shallow_groove', True))
scan_deep_pocket = bool(cfg.get('scan_deep_pocket', False))
scan_hydrophobic_patch = bool(cfg.get('scan_hydrophobic_patch', True))
scan_charged_patch = bool(cfg.get('scan_charged_patch', False))

MIN_POCKET_SIZE = int(cfg.get('min_pocket_size', 6))
HYDROPHOBIC_THRESHOLD = float(cfg.get('hydrophobic_threshold', 0.35))
CHARGED_THRESHOLD = float(cfg.get('charged_threshold', 0.30))

with open(YAML_PATH, 'r') as f:
    yaml_data = yaml.safe_load(f)
try:
    target_info = yaml_data['target_dict_cfg'][task_name]
    pdb_filename = target_info['target_filename'] + ".pdb"
    SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / target_info['source'] / pdb_filename
except KeyError:
    print(f"❌ Error: Target '{task_name}' not properly defined in YAML.")
    sys.exit(1)

RESULTS_CSV = BASE_DIR / 'screening_results' / task_name / f"{task_name}_automated_scans.csv"
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

KD_SCALE = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
            'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
            'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
CHARGE_SCALE = {'ARG': '+', 'LYS': '+', 'HIS': '+', 'ASP': '-', 'GLU': '-'}

# ---------------------------------------------------------
# 2. PDB Parsing & Grid Initialization
# ---------------------------------------------------------
ca_coords, res_ids, res_names = [], [], []
all_coords = []

with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[21] == target_chains:
            coord = [float(line[30:38]), float(line[38:46]), float(line[46:54])]
            all_coords.append(coord)
            if line[12:16].strip() == "CA":
                res_names.append(line[17:20].strip())
                res_ids.append(int(line[22:26].strip()))
                ca_coords.append(coord)

ca_coords = np.array(ca_coords)
res_ids = np.array(res_ids)
res_names = np.array(res_names)
all_coords = np.array(all_coords)
protein_kdtree = KDTree(ca_coords)

print("   -> Constructing 3D Voxel Cartography...")
GRID_RES = 1.0
PADDING = 15.0
coords_min = all_coords.min(axis=0) - PADDING
coords_max = all_coords.max(axis=0) + PADDING
grid_shape = np.ceil((coords_max - coords_min) / GRID_RES).astype(int)

protein_grid = np.zeros(grid_shape, dtype=bool)
for pt in all_coords:
    idx = np.round((pt - coords_min) / GRID_RES).astype(int)
    x, y, z = idx
    protein_grid[max(0, x-1):x+2, max(0, y-1):y+2, max(0, z-1):z+2] = True

# ---------------------------------------------------------
# 3. Virtual Shrink-Wrap & Bottleneck Severing
# ---------------------------------------------------------
print("   -> Deploying sturdy shrink-wrap boundary to define cavities...")
PROBE_RADIUS = 15.0
r = int(np.ceil(PROBE_RADIUS / GRID_RES))
zz, yy, xx = np.ogrid[-r:r+1, -r:r+1, -r:r+1]
sphere_struct = xx**2 + yy**2 + zz**2 <= r**2

wrapped_protein = binary_closing(protein_grid, structure=sphere_struct)
cavity_mask = wrapped_protein & ~protein_grid

print("   -> Severing narrow topological channels (EDT Bottleneck Erosion)...")
empty_space = ~protein_grid
edt = distance_transform_edt(empty_space) * GRID_RES

SEVERING_RADIUS = 2.5
isolated_cores = cavity_mask & (edt >= SEVERING_RADIUS)

if not np.any(isolated_cores):
    print("🛑 Error: No core cavities survived the bottleneck severing process.")
    sys.exit(0)

# ---------------------------------------------------------
# 4. Spatial Clustering of Isolated Cores
# ---------------------------------------------------------
print("   -> Clustering cleanly severed topological features...")
labeled_array, num_features = label(isolated_cores)
vertex_clusters = []

for i in range(1, num_features + 1):
    core_indices = np.argwhere(labeled_array == i)
    if len(core_indices) >= 40:
        voxel_coords = core_indices * GRID_RES + coords_min
        vertex_clusters.append(voxel_coords)

if not vertex_clusters:
    print("🛑 Scanning Complete: No disconnected core trenches found meeting volume limits.")
    sys.exit(0)

# ---------------------------------------------------------
# 5. Restoring Contact Surfaces (Dilation to CA atoms)
# ---------------------------------------------------------
pocket_results = []
categorized_findings = {'Grooves': [], 'Pockets': [], 'Hydrophobic_Patches': [], 'Charged_Patches': []}
count_groove = 0; count_pocket = 0; count_hydro = 0; count_charge = 0

for cluster_coords in vertex_clusters:
    neighbors = protein_kdtree.query_ball_point(cluster_coords, r=6.5)

    res_indices = set()
    for nl in neighbors: res_indices.update(nl)

    if len(res_indices) < MIN_POCKET_SIZE:
        continue

    cluster_res_indices = list(res_indices)
    c_res_names = res_names[cluster_res_indices]
    total = len(cluster_res_indices)

    topo_type = "Deep Pocket" if total > 25 else "Shallow Groove"

    hydro_count = sum(1 for name in c_res_names if KD_SCALE.get(name, 0) > 0)
    pos_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '+')
    neg_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '-')
    charge_count = pos_count + neg_count

    hydro_ratio = hydro_count / total
    charge_ratio = charge_count / total

    is_hydro = hydro_ratio >= HYDROPHOBIC_THRESHOLD
    is_charge = charge_ratio >= CHARGED_THRESHOLD

    c_res_ids = res_ids[cluster_res_indices]
    patch_str = ",".join(map(str, sorted(c_res_ids)))
    pymol_sel = "+".join(map(str, sorted(c_res_ids)))

    site_id = f"Site_{len(pocket_results)+1}"
    pocket_results.append({
        'Site_ID': site_id, 'Topology': topo_type, 'Residues': total,
        'Hydrophobic_Ratio': round(hydro_ratio, 3), 'Charged_Ratio': round(charge_ratio, 3),
        'Hotspot_Sequence': patch_str
    })

    # Added raw c_res_ids list to tuple for py3Dmol rendering
    if topo_type == "Shallow Groove" and scan_shallow_groove:
        count_groove += 1
        spec_id = f"Groove_{count_groove}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color orange, {spec_id}; show surface, {spec_id}"
        categorized_findings['Grooves'].append((spec_id, cmd, list(c_res_ids)))

    if topo_type == "Deep Pocket" and scan_deep_pocket:
        count_pocket += 1
        spec_id = f"Pocket_{count_pocket}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color magenta, {spec_id}; show surface, {spec_id}"
        categorized_findings['Pockets'].append((spec_id, cmd, list(c_res_ids)))

    if is_hydro and scan_hydrophobic_patch:
        count_hydro += 1
        spec_id = f"Hydro_Patch_{count_hydro}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color yellow, {spec_id}; show surface, {spec_id}"
        categorized_findings['Hydrophobic_Patches'].append((spec_id, cmd, list(c_res_ids)))

    if is_charge and scan_charged_patch:
        count_charge += 1
        spec_id = f"Charge_Patch_{count_charge}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color cyan, {spec_id}; show surface, {spec_id}"
        categorized_findings['Charged_Patches'].append((spec_id, cmd, list(c_res_ids)))

# ---------------------------------------------------------
# 6. Printed Output
# ---------------------------------------------------------
print("\n" + "="*70)
print(f"Summary: {count_groove} Grooves, {count_pocket} Pockets, {count_hydro} Hydro Patches, {count_charge} Charged Patches.")
print("="*70 + "\n")

if categorized_findings['Grooves']:
    print("Grooves:")
    for t_id, cmd, _ in categorized_findings['Grooves']:
        print(f"  {t_id}:\n  {cmd}\n")

if categorized_findings['Pockets']:
    print("Pockets:")
    for t_id, cmd, _ in categorized_findings['Pockets']:
        print(f"  {t_id}:\n  {cmd}\n")

if categorized_findings['Hydrophobic_Patches']:
    print("Patches (Hydrophobic):")
    for t_id, cmd, _ in categorized_findings['Hydrophobic_Patches']:
        print(f"  {t_id}:\n  {cmd}\n")

if categorized_findings['Charged_Patches']:
    print("Patches (Charged):")
    for t_id, cmd, _ in categorized_findings['Charged_Patches']:
        print(f"  {t_id}:\n  {cmd}\n")

if pocket_results:
    pd.DataFrame(pocket_results).to_csv(RESULTS_CSV, index=False)

# ---------------------------------------------------------
# 7. In-Notebook 3D Morphological Rendering
# ---------------------------------------------------------
if SOURCE_PDB_PATH.exists() and count_groove > 0:
    print("\n   -> Rendering 3D Topographical Map (Gray Mesh for Protein, Orange Surface for Grooves)...")

    with open(SOURCE_PDB_PATH, 'r') as f:
        pdb_data = f.read()

    viewer = py3Dmol.view(width=800, height=500)
    viewer.addModel(pdb_data, 'pdb')

    # Render main protein as gray cartoon and gray mesh (wireframe surface)
    viewer.setStyle({'chain': target_chains}, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, {'chain': target_chains})

    # Iterate through found grooves and render them as orange solid surfaces
    for spec_id, cmd, resi_list in categorized_findings['Grooves']:
        str_resis = [str(r) for r in resi_list]
        groove_sel = {'chain': target_chains, 'resi': str_resis}

        # Color the backbone slightly for better internal contrast
        viewer.setStyle(groove_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        # Overlay the solid orange surface
        viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, groove_sel)

    viewer.zoomTo()
    viewer.show()
elif count_groove == 0:
    print("\n   -> Visualization skipped: No grooves identified meeting the designated criteria.")

# Purpose: Analyze spatial topologies to locate protein interaction sites, generating Pymol commands and in-notebook 3D mesh/surface renderings.
# Upstream Code: Global configuration settings.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 13:03 EDT.
# Changed Lines:
# - Inserted Lines 11-12: Imported py3Dmol and IPython display modules.
# - Modified Lines 160-179: Appended the raw list of residue integers `list(c_res_ids)` to the `categorized_findings` dictionary values.
# - Inserted Lines 212-237: Added Section 7 logic. Constructed interactive spatial viewer to render target chains as gray wireframe (mesh) and highlighted grooves as orange solid surfaces utilizing boolean residue mapping.

🧬 Initializing Morphological Bottleneck Severing Engine (Optimized for Split-Barrels)...
   -> Constructing 3D Voxel Cartography...
   -> Deploying sturdy shrink-wrap boundary to define cavities...
   -> Severing narrow topological channels (EDT Bottleneck Erosion)...
   -> Clustering cleanly severed topological features...

Summary: 1 Grooves, 0 Pockets, 0 Hydro Patches, 0 Charged Patches.

Grooves:
  Groove_1:
  select Groove_1, chain A and resi 39+40+41+73+74+200+201+202+203+204; color orange, Groove_1; show surface, Groove_1


   -> Rendering 3D Topographical Map (Gray Mesh for Protein, Orange Surface for Grooves)...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# # Cell_5b_Automated_Groove_Scanning_Engine.py
# # Requirement: Paradigm shift to Opposing Ridges Void Midpoint (ORVM) algorithm. Specifically targets the user's request to exclusively highlight the two parallel ridges forming a trench. It identifies pairs of sequentially distant atoms separated by an 8.5-13.5A continuous void, meticulously isolating the "left bank" and "right bank" of missing beta-strands.

# import os, json, sys, yaml
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from scipy.spatial import KDTree

# print("🔍 Initializing Opposing Ridges Void Midpoint (ORVM) Engine...")

# # ---------------------------------------------------------
# # 1. Load Configurations & YAML Interface
# # ---------------------------------------------------------
# BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
# SETTING_FILE = BASE_DIR / 'internal_settings.json'
# YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

# if not SETTING_FILE.exists() or not YAML_PATH.exists():
#     print("❌ Error: Required configuration files not found.")
#     sys.exit(1)

# with open(SETTING_FILE, 'r') as f:
#     cfg = json.load(f)

# task_name = cfg['task_name']
# target_chains = cfg['target_chains']

# scan_shallow_groove = bool(cfg.get('scan_shallow_groove', True))
# scan_deep_pocket = bool(cfg.get('scan_deep_pocket', False))
# scan_hydrophobic_patch = bool(cfg.get('scan_hydrophobic_patch', True))
# scan_charged_patch = bool(cfg.get('scan_charged_patch', False))

# MIN_POCKET_SIZE = int(cfg.get('min_pocket_size', 6))
# HYDROPHOBIC_THRESHOLD = float(cfg.get('hydrophobic_threshold', 0.35))
# CHARGED_THRESHOLD = float(cfg.get('charged_threshold', 0.30))

# with open(YAML_PATH, 'r') as f:
#     yaml_data = yaml.safe_load(f)
# try:
#     target_info = yaml_data['target_dict_cfg'][task_name]
#     pdb_filename = target_info['target_filename'] + ".pdb"
#     SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / target_info['source'] / pdb_filename
# except KeyError:
#     print(f"❌ Error: Target '{task_name}' not properly defined in YAML.")
#     sys.exit(1)

# RESULTS_CSV = BASE_DIR / 'screening_results' / task_name / f"{task_name}_automated_scans.csv"
# RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

# KD_SCALE = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
#             'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
#             'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
# CHARGE_SCALE = {'ARG': '+', 'LYS': '+', 'HIS': '+', 'ASP': '-', 'GLU': '-'}

# # ---------------------------------------------------------
# # 2. PDB Parsing & Coordinate Extraction
# # ---------------------------------------------------------
# ca_coords, res_ids, res_names = [], [], []

# with open(SOURCE_PDB_PATH, 'r') as f:
#     for line in f:
#         if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
#             res_names.append(line[17:20].strip())
#             res_ids.append(int(line[22:26].strip()))
#             ca_coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

# ca_coords = np.array(ca_coords)
# res_ids = np.array(res_ids)
# res_names = np.array(res_names)
# protein_kdtree = KDTree(ca_coords)

# # ---------------------------------------------------------
# # 3. Opposing Ridges Void Midpoint (ORVM) Algorithm
# # ---------------------------------------------------------
# print("   -> Scanning for opposing structural ridges separated by a continuous void...")

# # Extract all CA atom pairs within a 14.0A distance
# pairs = protein_kdtree.query_pairs(r=14.0)
# valid_pairs = []
# midpoints = []

# for i, j in pairs:
#     # a. Physical cross-trench distance: The gap of a missing Beta strand typically maintains a bank-to-bank distance between 8.5A and 13.5A
#     dist = np.linalg.norm(ca_coords[i] - ca_coords[j])
#     if dist < 8.5 or dist > 13.5:
#         continue

#     # b. Sequence disconnection filter: Ensure the pairs belong to spatially adjacent but sequentially distant strands, avoiding continuous backbone
#     if np.abs(res_ids[i] - res_ids[j]) < 15:
#         continue

#     # c. Vacuum midpoint generation: Calculate the spatial geometric center between the pair
#     M = (ca_coords[i] + ca_coords[j]) / 2.0

#     # d. Midpoint clearance validation: The midpoint must exist in a void (at least 3.8A away from any CA backbone atom)
#     nearest_dist, _ = protein_kdtree.query(M, k=1)
#     if nearest_dist < 3.8:
#         continue

#     # e. Surface vs Core Filter:
#     # A true surface groove's midpoint is surrounded only by the floor and sides (10-32 atoms).
#     # If trapped inside a chromophore cavity, it would be surrounded by the entire barrel from all directions (> 35 atoms).
#     neighbors_count = len(protein_kdtree.query_ball_point(M, r=12.0))
#     if neighbors_count < 10 or neighbors_count > 32:
#         continue

#     valid_pairs.append((i, j))
#     midpoints.append(M)

# midpoints = np.array(midpoints)

# if len(midpoints) == 0:
#     print("🛑 Scanning Complete: No opposing ridges across a valid topological trench were found.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 4. Clustering Validated Midpoints into Continuous Trenches
# # ---------------------------------------------------------
# print("   -> Stitching cross-trench voids to isolate distinct binding grooves...")
# midpoint_kdtree = KDTree(midpoints)

# # If multiple vacuum midpoints are exceedingly close (4.5A), they belong to the same continuous groove
# mp_pairs = midpoint_kdtree.query_pairs(r=4.5)

# adj_list = {i: set() for i in range(len(midpoints))}
# for i, j in mp_pairs:
#     adj_list[i].add(j); adj_list[j].add(i)

# visited = set()
# final_ridge_clusters = []

# for i in range(len(midpoints)):
#     if i not in visited:
#         queue = [i]
#         component = []
#         while queue:
#             node = queue.pop(0)
#             if node not in visited:
#                 visited.add(node)
#                 component.append(node)
#                 queue.extend(list(adj_list[node] - visited))

#         # A valid biological cleft will generate at least 8 valid vacuum midpoint pairs
#         if len(component) >= 8:
#             # Extract all atom indices from the left and right banks contributing to this groove
#             ridge_indices = set()
#             for idx in component:
#                 u, v = valid_pairs[idx]
#                 ridge_indices.add(u)
#                 ridge_indices.add(v)
#             final_ridge_clusters.append(list(ridge_indices))

# if not final_ridge_clusters:
#     print("🛑 Scanning Complete: Detected opposing ridges were too fragmented to form a continuous groove.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 5. Categorization & Annotation
# # ---------------------------------------------------------
# pocket_results = []
# categorized_findings = {'Grooves': [], 'Pockets': [], 'Hydrophobic_Patches': [], 'Charged_Patches': []}

# count_groove = 0; count_pocket = 0; count_hydro = 0; count_charge = 0

# for cluster in final_ridge_clusters:
#     c_res_names = res_names[cluster]
#     total = len(cluster)

#     # This pairing algorithm purely extracts the elevated opposing ridges on both sides
#     topo_type = "Deep Pocket" if total > 35 else "Shallow Groove"

#     hydro_count = sum(1 for name in c_res_names if KD_SCALE.get(name, 0) > 0)
#     pos_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '+')
#     neg_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '-')
#     charge_count = pos_count + neg_count

#     hydro_ratio = hydro_count / total
#     charge_ratio = charge_count / total

#     is_hydro = hydro_ratio >= HYDROPHOBIC_THRESHOLD
#     is_charge = charge_ratio >= CHARGED_THRESHOLD

#     c_res_ids = res_ids[cluster]
#     patch_str = ",".join(map(str, sorted(c_res_ids)))
#     pymol_sel = "+".join(map(str, sorted(c_res_ids)))

#     site_id = f"Site_{len(pocket_results)+1}"
#     pocket_results.append({
#         'Site_ID': site_id, 'Topology': topo_type, 'Residues': total,
#         'Hydrophobic_Ratio': round(hydro_ratio, 3), 'Charged_Ratio': round(charge_ratio, 3),
#         'Hotspot_Sequence': patch_str
#     })

#     if topo_type == "Shallow Groove" and scan_shallow_groove:
#         count_groove += 1
#         spec_id = f"Groove_{count_groove}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color orange, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Grooves'].append((spec_id, cmd))

#     if topo_type == "Deep Pocket" and scan_deep_pocket:
#         count_pocket += 1
#         spec_id = f"Pocket_{count_pocket}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color magenta, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Pockets'].append((spec_id, cmd))

#     if is_hydro and scan_hydrophobic_patch:
#         count_hydro += 1
#         spec_id = f"Hydro_Patch_{count_hydro}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color yellow, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Hydrophobic_Patches'].append((spec_id, cmd))

#     if is_charge and scan_charged_patch:
#         count_charge += 1
#         spec_id = f"Charge_Patch_{count_charge}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color cyan, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Charged_Patches'].append((spec_id, cmd))

# # ---------------------------------------------------------
# # 6. Printed Output
# # ---------------------------------------------------------
# print("\n" + "="*70)
# print(f"Summary: {count_groove} Grooves, {count_pocket} Pockets, {count_hydro} Hydro Patches, {count_charge} Charged Patches.")
# print("="*70 + "\n")

# if categorized_findings['Grooves']:
#     print("Grooves:")
#     for t_id, cmd in categorized_findings['Grooves']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Pockets']:
#     print("Pockets:")
#     for t_id, cmd in categorized_findings['Pockets']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Hydrophobic_Patches']:
#     print("Patches (Hydrophobic):")
#     for t_id, cmd in categorized_findings['Hydrophobic_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Charged_Patches']:
#     print("Patches (Charged):")
#     for t_id, cmd in categorized_findings['Charged_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if pocket_results:
#     pd.DataFrame(pocket_results).to_csv(RESULTS_CSV, index=False)

# # ==============================================================================
# # Update Log:
# # Purpose: Deployed the Opposing Ridges Void Midpoint (ORVM) algorithm specifically designed to fulfill the user's intent to highlight *only* the two parallel ridges flanking a trench. It calculates geometric midpoints between sequentially distant atoms (gap 8.5-13.5A). It strictly guarantees void clearance (no CA within 3.8A) and rejects central barrel voids by limiting neighborhood counts (10-32 atoms within 12A). By extracting only the CA pairs generating these clustered void midpoints, it provides an exquisite isolation of trench lips with zero bleed to the floor or surrounding convex topology.
# # Upstream Code: Complete algorithmic refactor of Section 3-4.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-03 22:15 EDT.
# # Changed Lines:
# # - Lines 85-115: Implemented Void Midpoint generation and verification logic.
# # - Lines 103: `nearest_dist < 3.8` enforces perfect physical emptiness between the two ridges.
# # - Lines 110: `neighbors_count < 10 or neighbors_count > 32` prevents central GFP barrel hollows from registering as surface grooves.
# # - Lines 142-146: Extracted only the original `(u, v)` pair atom indices that formed the valid trench void, directly producing the two distinct mountain ridges.
# # ==============================================================================

In [ ]:

# =================================================================
# # Cell_6_Automated Pipeline for GFP Natural Substrate Benchmarking, Metrics Dashboard, and 3D Visualization
# =================================================================

import os, sys, subprocess, shutil, gc, torch, time
import pandas as pd
from pathlib import Path
import py3Dmol

#--- [Configuration Section] ---
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGET_PDB_NAME = "GFP_1_10.pdb"
BINDER_PDB_NAME = "GFP_11.pdb"
TARGET_TASK = "GFP"
RUN_NAME = "Run_Native_Baseline"
JOB_ID = 0

# --- [Path Pre-check] ---
target_pdb_path = BASE_DIR / "targets" / TARGET_PDB_NAME
binder_pdb_path = BASE_DIR / "targets" / BINDER_PDB_NAME

if not target_pdb_path.exists() or not binder_pdb_path.exists():
    print(f"Error: Specified PDB file could not be located.")
    sys.exit(1)

inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{TARGET_TASK}_{RUN_NAME}'
if inference_dir.exists(): shutil.rmtree(inference_dir)
inference_dir.mkdir(parents=True, exist_ok=True)

mock_design_id = f"job_{JOB_ID}_native_baseline"
mock_job_dir = inference_dir / mock_design_id
mock_job_dir.mkdir(parents=True, exist_ok=True)

# --- [Core Patch 1: Generic Parameter] ---
utils_file = BASE_DIR / 'src/proteinfoundation/metrics/metric_utils.py'
if utils_file.exists():
    with open(utils_file, 'r') as f: content = f.read()
    if "def replace_seq_in_generated_pdb(" in content and "def _disabled_replace" not in content:
        shutil.copy(utils_file, str(utils_file) + ".original")
        safe_override = (
            "def replace_seq_in_generated_pdb(*args, **kwargs):\n"
            "    import shutil, os\n"
            "    for val in list(args) + list(kwargs.values()):\n"
            "        if isinstance(val, str) and val.endswith('.pdb'):\n"
            "            out_path = val.replace('.pdb', '_updated.pdb')\n"
            "            if not os.path.exists(out_path): shutil.copy(val, out_path)\n"
            "    return\n\n"
            "def _disabled_replace("
        )
        with open(utils_file, 'w') as f: f.write(content.replace("def replace_seq_in_generated_pdb(", safe_override))

# --- [Core Patch 2: Resolving Misaligned Index and Out-of-Bounds Bug] ---
binder_metrics_file = BASE_DIR / 'src/proteinfoundation/metrics/binder_metrics.py'
if binder_metrics_file.exists():
    with open(binder_metrics_file, 'r') as f:
        bm_content = f.read()
    buggy_line = 'interface_seq = "".join([sequence[i] for i in interface_residues])'
    safe_line = 'interface_seq = "".join([sequence[i] for i in interface_residues if i < len(sequence)])'
    if buggy_line in bm_content:
        shutil.copy(binder_metrics_file, str(binder_metrics_file) + ".bak")
        with open(binder_metrics_file, 'w') as f:
            f.write(bm_content.replace(buggy_line, safe_line))
        print("Patch applied: Fixed index out-of-bounds crash in interface_seq.")

#--- [PDB Assembly (Chain A+B)] ---
target_pdb_eval_path = mock_job_dir / f"{mock_design_id}.pdb"
atom_idx = 1
with open(target_pdb_eval_path, 'w') as f_out:
    for chain, p in [('A', target_pdb_path), ('B', binder_pdb_path)]:
        with open(p, 'r') as f_in:
            for line in f_in:
                if line.startswith(("ATOM", "HETATM")):
                    f_out.write(line[:6] + f"{atom_idx:>5}" + line[11:21] + chain + line[22:])
                    atom_idx += 1
        f_out.write("TER\n")
    f_out.write("END\n")

# --- [4. VRAM Cleanup & Function Execution] ---
def run_PF_command(command_str):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    env = os.environ.copy()
    env.update({'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'TF_FORCE_GPU_ALLOW_GROWTH': 'true'})

    full_command = f"source env.sh && {command_str}"
    process = subprocess.Popen(full_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), env=env)
    for line in process.stdout: print(line, end='', flush=True)
    return process.wait()

base_args = (f"--config-path {BASE_DIR}/configs --config-name search_binder_local_pipeline "
             f"++job_id={JOB_ID} ++generation.task_name={TARGET_TASK} ++run_name={RUN_NAME} "
             f"++generation.dataloader.dataset.conditional_features.0.pdb_path={target_pdb_path} "
             f"++generation.dataloader.batch_size=1 ++hydra.job.chdir=False ++eval_nmodels=1")

print(f"\n AlphaFold2 Evaluation Started...")
ret = run_PF_command(f"python3 -m proteinfoundation.evaluate {base_args} ++run_evaluate=True ++eval_njobs=1")

if ret == 0:
    run_PF_command(f"python3 -m proteinfoundation.analyze {base_args} ++run_analyze=True")
    eval_out_dir = BASE_DIR / 'evaluation_results' / f'search_binder_local_pipeline_{TARGET_TASK}_{RUN_NAME}'
    csv_files = list(eval_out_dir.glob("binder_results_*.csv"))

    if csv_files:

        df = pd.read_csv(max(csv_files, key=lambda x: x.stat().st_size))
        d = df.iloc[0]


        print("\n🏆 Gold Standard Baseline (10AA Core Complex Reference):")
        header_format = "{:<8} | {:<15} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
        print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD"))
        print("-" * 85)
        print(header_format.format(
            "CORE", "GFP_11_CORE",
            f"{d.get('self_complex_max_ipSAE', 0):.4f}",
            f"{d.get('self_complex_pTM', 0):.4f}",
            f"{d.get('self_complex_i_pTM', 0):.4f}",
            f"{d.get('self_complex_pLDDT', 0):.2f}",
            f"{d.get('self_binder_scRMSD_ca', 0):.2f}Å"
        ))
        print("-" * 85)


        print("\n🎯 --- GFP_1_10.pdb + GFP_11.pdb ---")
        for index, row in df.iterrows():

            print(f"  > i_pSAE: {row.get('self_complex_max_ipSAE', row.get('complex_i_pAE', 'N/A'))}")
            print(f"  > pTM            : {row.get('self_complex_pTM', row.get('complex_pTM', 'N/A'))}")
            print(f"  > ipTM           : {row.get('self_complex_i_pTM', row.get('complex_ipTM', 'N/A'))}")
            print(f"  > pLDDT   : {row.get('self_complex_pLDDT', row.get('complex_pLDDT', 'N/A'))}")
            print(f"  > Binder scRMSD  : {row.get('self_binder_scRMSD_ca', row.get('binder_scRMSD_ca', 'N/A'))}")
            print("\n Complete Data Sheet：")
            print(df.T)

# --- Save CSV outside the loop, alongside the PDB dir ---
        target_csv_path = mock_job_dir / f"{mock_design_id}_metrics.csv"
        df.to_csv(target_csv_path, header=True)
        print(f"\n✅ Data Sheet saved alongside PDB at: {target_csv_path.relative_to(BASE_DIR)}")
   # --- [6. Complex Visualization] ---

        import pandas as pd

        COMPLEX_PDB_PATH = eval_out_dir / mock_design_id / "AF2" / f"{mock_design_id}_self_seq_0_model1.pdb"
        SCAN_CSV_PATH = BASE_DIR / 'screening_results' / TARGET_TASK / f"{TARGET_TASK}_automated_scans.csv"

        if COMPLEX_PDB_PATH.exists():
            print(f"\n   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...")
            with open(COMPLEX_PDB_PATH, 'r') as f:
                pdb_data = f.read()

            viewer = py3Dmol.view(width=800, height=500)
            viewer.addModel(pdb_data, 'pdb')

            # 1. Target Base: Gray cartoon and gray mesh (wireframe surface)
            target_sel = {'chain': 'A'}
            viewer.setStyle(target_sel, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
            viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, target_sel)

            # 2. Target Grooves: Parse CSV and apply Orange surface
            if SCAN_CSV_PATH.exists():
                df_scan = pd.read_csv(SCAN_CSV_PATH)
                groove_df = df_scan[df_scan['Topology'] == 'Shallow Groove']
                groove_resis = []
                for _, row in groove_df.iterrows():
                    # Handle potential float/nan issues and split comma-separated strings
                    seq_val = str(row.get('Hotspot_Sequence', ''))
                    if seq_val and seq_val != 'nan':
                        groove_resis.extend(seq_val.split(','))

                if groove_resis:
                    groove_sel = {'chain': 'A', 'resi': groove_resis}
                    viewer.setStyle(groove_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
                    viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, groove_sel)

            # 3. Binder: Bright green cartoon and solid surface
            binder_sel = {'chain': 'B'}
            viewer.setStyle(binder_sel, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
            viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, binder_sel)

            viewer.zoomTo()
            viewer.show()

# Purpose: Merge upstream topological data with the structural evaluation viewer. Reads the pre-calculated CSV to render specific target residues as orange grooves, maintaining gray wireframes for the rest of the target and bright green for the binder.
# Upstream Code: Relies on Cell 11 for the generation of the automated_scans.csv file.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 19:49 EDT.
# Changed Lines:
# * Added pandas import and SCAN_CSV_PATH resolution.
# * Inserted logic block #2 to read CSV, filter for 'Shallow Groove', extract 'Hotspot_Sequence', and apply py3Dmol styling dynamically to those specific residues on chain A.
        else:
            print(f"\n   -> ⚠️ Visualization skipped: Could not find complex PDB files in : {COMPLEX_PDB_PATH}")

else:
    print(f"\n❌ Evaluation failed with code: {ret}。")

Patch applied: Fixed index out-of-bounds crash in interface_seq.

 AlphaFold2 Evaluation Started...
Complexa environment initialized for uv runtime.
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/einops/einops.py:737: SyntaxWarning: invalid escape sequence '\s'
  \sum_{c, d, g} x[a, b, c] * y[c, b, d] * z[a, g, k]
2026-04-07 02:42:09.158 | WARNING  | proteinfoundation.evaluation.binder_eval:<module>:53 - TMOL import failed: No module named 'tmol'. TMOL metrics will return NaN.
/usr/local/lib/python3.12/dist-packages/prody/atomic/flags.py:532: SyntaxWarning: invalid escape sequence '\ '
  \                            *PDB*  *Source*  *Conflict*
/usr/local/lib/python3.12/dist-packages/prody/atomic/select.py:203: SyntaxWarning: invalid escape sequence '\|'
  **Special characters** 

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Section 4: Threshold-Driven Autonomous Screening

In [ ]:
# Cell_7b_Autonomous_Generation_Loop.py
# Requirement: Execute an automated loop for high-throughput generation and evaluation.
# ENHANCEMENT: Explicitly output seed number for every cycle for debugging.
# Fixed duplicate logging / leaderboard duplication bug.

import os, json, sys, time, subprocess, csv, datetime, shutil, math, random, gc
from pathlib import Path
import torch

# ---------------------------------------------------------
# 1. Load Persistent Configuration
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please navigate to Section 0 and configure parameters.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_threshold = float(cfg['target_threshold'])
target_success_count = int(cfg['target_success_count'])
designs_per_loop = int(cfg['designs_per_loop'])
max_iterations = int(cfg['max_iterations'])
batch_size = int(cfg['batch_size'])

print(f"🤖 Auto-Pilot Engine Activated for Target: [{task_name}]")
print(f"🎯 Objective Strategy: Capture {target_success_count} qualified hits surpassing threshold {target_threshold}.")
print(f"🔄 Loop Matrix: Designing {designs_per_loop} structures per cycle (Maximum threshold: {max_iterations} cycles).")

# ---------------------------------------------------------
# 2. Tracking Variables, Logging & Master Hierarchy Setup
# ---------------------------------------------------------
ALL_DESIGNS_POOL = []          # Global pool (will deduplicate by design_id + dir)
GLOBAL_HITS = []

SCREENING_RESULTS_DIR = BASE_DIR / 'screening_results' / task_name
SCREENING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
master_log_path = SCREENING_RESULTS_DIR / "autopilot_master_log.csv"

SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not SOURCE_PDB_PATH.exists():
    print(f"❌ Error: Required source PDB structure not found at {SOURCE_PDB_PATH}. Run Section 2 first.")
    sys.exit(1)

SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"AutoPilot_{task_name}_{SESSION_TIMESTAMP}"
SESSION_MASTER_DIR = BASE_DIR / 'assets' / 'target_data' / SESSION_NAME
SESSION_MASTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Master Session Directory established at: {SESSION_MASTER_DIR.relative_to(BASE_DIR)}")

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m}m {s}s"

env = os.environ.copy()
env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})



# ---------------------------------------------------------
# 4. High-Throughput Engine Execution Loop
# ---------------------------------------------------------
global_start_time = time.time()
current_iteration = 0

while current_iteration < max_iterations and len(GLOBAL_HITS) < target_success_count:
    current_iteration += 1
    loop_start_time = time.time()
    batch_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # === NEW: Generate and explicitly print seed for every cycle ===
    current_seed = random.randint(1, 1000000)
    print(f"\n🔑 DEBUG: Seed used for Cycle {current_iteration} = {current_seed}")

    current_run_name = f"{SESSION_NAME}_Cycle{current_iteration}"
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / current_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 Auto-Pilot Cycle Iteration [{current_iteration}/{max_iterations}] Commencing...")
    print(f"📂 Cycle Sandbox: {RUN_ISOLATED_DIR.relative_to(SESSION_MASTER_DIR)}")
    print(f"=======================================================")

    num_batches = math.ceil(designs_per_loop / batch_size)
    total_designs = num_batches * batch_size

    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={current_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={total_designs} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=True ++run_analyze=True ++seed={current_seed}"
    )

    print(f"🔥 Transmitting execution command to the underlying engine (Seed: {current_seed})...")

    # ETA monitoring (unchanged)
    iter_start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n🕒 Cycle {current_iteration} Start Time: {iter_start_str}")
    if current_iteration > 1:
        avg_cycle_time = (time.time() - global_start_time) / (current_iteration - 1)
        remaining_cycles = max_iterations - current_iteration + 1
        eta_seconds = avg_cycle_time * remaining_cycles
        eta_str = (datetime.datetime.now() + datetime.timedelta(seconds=eta_seconds)).strftime("%Y-%m-%d %H:%M:%S")
        print(f"⏳ Estimated Time of Completion (ETA) for max cycles: {eta_str}")
    else:
        print(f"⏳ Estimated Time of Completion (ETA): Calculating after first cycle...")

    # Run pipeline
    process = subprocess.Popen(
        f"source env.sh && {cmd_str}",
        env=env, shell=True, executable='/bin/bash', cwd=str(BASE_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    ret_code = process.returncode

    if ret_code != 0:
        print(f"❌ Critical Warning: Underlying generation engine crashed (Error Code {ret_code})!")
        time.sleep(3)
        continue

    # ---------------------------------------------------------
    # 5. Report Parsing & Dynamic Leaderboard (deduplication added)
    # ---------------------------------------------------------
    inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{current_run_name}'
    valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

    if valid_csvs:
        target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
        with open(target_csv, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            new_entries_this_cycle = 0
            for i, row in enumerate(reader):
                try:
                    ipsae_val = float(row.get('af2folding_max_ipsae', row.get('self_complex_max_ipAE', row.get('complex_i_pAE', -999.0))))
                    ptm_val = float(row.get('af2folding_ptm_log', row.get('self_complex_pTM', row.get('complex_pTM', -999.0))))
                    iptm_val = float(row.get('af2folding_i_ptm_log', row.get('self_complex_i_pTM', row.get('complex_ipTM', -999.0))))
                    plddt_val = float(row.get('af2folding_plddt', row.get('self_complex_pLDDT', row.get('complex_pLDDT', -999.0))))
                    rmsd_val = float(row.get('af2folding_rmsd', row.get('self_binder_scRMSD_ca', row.get('binder_scRMSD_ca', -999.0))))
                    total_reward = float(row.get('total_reward', -999.0))
                    d_id = row.get('design_id', '').strip() or f"Design_{row.get('pdb_index', i)}"

                    # Deduplication key: (design_id, inference_dir)
                    design_key = (d_id, str(inference_dir))
                    if any((d['id'], str(d['dir'])) == design_key for d in ALL_DESIGNS_POOL):
                        continue  # skip duplicate

                    design_entry = {
                        'id': d_id, 'score': ipsae_val, 'ptm': ptm_val, 'iptm': iptm_val,
                        'plddt': plddt_val, 'rmsd': rmsd_val, 'total_reward': total_reward,
                        'batch_num': current_iteration, 'idx_in_batch': i + 1,
                        'dir': inference_dir, 'loop': current_iteration
                    }
                    ALL_DESIGNS_POOL.append(design_entry)
                    new_entries_this_cycle += 1

                    if ipsae_val >= target_threshold:
                        if not any(h['id'] == d_id and str(h['dir']) == str(inference_dir) for h in GLOBAL_HITS):
                            GLOBAL_HITS.append(design_entry)
                except Exception:
                    pass

        # Write to master log (once per cycle)
        file_exists = master_log_path.exists()
        with open(master_log_path, 'a', encoding='utf-8', newline='') as f_log:
            fieldnames = ['session_loop', 'timestamp', 'design_id', 'max_ipSAE', 'pTM', 'iPTM', 'pLDDT', 'RMSD', 'total_reward', 'source_path']
            writer = csv.DictWriter(f_log, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            for entry in [d for d in ALL_DESIGNS_POOL if d['loop'] == current_iteration]:
                writer.writerow({
                    'session_loop': entry['loop'], 'timestamp': batch_timestamp, 'design_id': entry['id'],
                    'max_ipSAE': f"{entry['score']:.4f}", 'pTM': f"{entry['ptm']:.4f}",
                    'iPTM': f"{entry['iptm']:.4f}", 'pLDDT': f"{entry['plddt']:.2f}",
                    'RMSD': f"{entry['rmsd']:.2f}", 'total_reward': f"{entry['total_reward']:.4f}",
                    'source_path': str(entry['dir'].relative_to(BASE_DIR))
                })

        print(f"📝 {new_entries_this_cycle} new designs appended to autopilot_master_log.csv (Seed: {current_seed})")
    else:
        print("⚠️ Warning: Pipeline succeeded, but no valid CSV output files were found.")

    # ---------------------------------------------------------
    # 6. Cycle Summary & Execution Metrics Display
    # ---------------------------------------------------------
    cycle_duration = format_time(time.time() - loop_start_time)
    print(f"\n⏱️ Cycle {current_iteration} Operational Duration: {cycle_duration}")
    print(f"📈 Verified Candidates Captured (ipSAE >= {target_threshold}): {len(GLOBAL_HITS)} / {target_success_count}")

    ALL_DESIGNS_POOL.sort(key=lambda x: x['score'], reverse=True)

    print("\n🏆 Live Top 10 Design Roster (Global Index prioritized by max_ipSAE):")
    header_format = "{:<5} | {:<12} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<6} | {:<6}"
    print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD", "Reward", "Batch", "Index"))
    print("-" * 105)

    for rank, d in enumerate(ALL_DESIGNS_POOL[:10], 1):
        print(header_format.format(
            rank, d['id'][:12], f"{d['score']:.4f}", f"{d['ptm']:.4f}", f"{d['iptm']:.4f}",
            f"{d['plddt']:.2f}", f"{d['rmsd']:.2f}Å", f"{d['total_reward']:.4f}",
            str(d['batch_num']), str(d['idx_in_batch'])
        ))
    print("-" * 105)

    # VRAM cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    if len(GLOBAL_HITS) >= target_success_count:
        print(f"\n🎉 Threshold Attained! Successfully assimilated {target_success_count} rigorously vetted structural models.")
        break

    if current_iteration < max_iterations:
        print("🧹 Buffer memory cleared. Entering brief thermal cool-down before subsequent cycle initiation...")
        time.sleep(5)

# ---------------------------------------------------------
# 7. Final Execution Post-Mortem Output
# ---------------------------------------------------------
print(f"\n🛑 Auto-Pilot execution state terminated.")
if len(GLOBAL_HITS) >= target_success_count:
    print(f"🏆 OBJECTIVE COMPLETED: Hit quota satisfied yielding {len(GLOBAL_HITS)} high-affinity models.")
else:
    print(f"⚠️ MAXIMUM ITERATIONS TRIGGERED: Algorithm halted with {len(GLOBAL_HITS)} viable candidates isolated over {current_iteration} execution cycles.")

print(f"📂 Persistent Master Log registry accessible via: {master_log_path}")

🤖 Auto-Pilot Engine Activated for Target: [GFP]
🎯 Objective Strategy: Capture 10 qualified hits surpassing threshold 0.85.
🔄 Loop Matrix: Designing 36 structures per cycle (Maximum threshold: 5 cycles).
📂 Master Session Directory established at: assets/target_data/AutoPilot_GFP_Y2026_M04_D08_H12_M12_S02

🔑 DEBUG: Seed used for Cycle 1 = 30805

🔄 Auto-Pilot Cycle Iteration [1/5] Commencing...
📂 Cycle Sandbox: AutoPilot_GFP_Y2026_M04_D08_H12_M12_S02_Cycle1
🔥 Transmitting execution command to the underlying engine (Seed: 30805)...

🕒 Cycle 1 Start Time: 2026-04-08 12:12:02
⏳ Estimated Time of Completion (ETA): Calculating after first cycle...
Complexa environment initialized for uv runtime.

  🧬 Complexa Binder Design Pipeline
  Config:       configs/search_binder_local_pipeline.yaml
  Target:       GFP
  Run:          AutoPilot_GFP_Y2026_M04_D08_H12_M12_S02_Cycle1
  GPUs:         1
  Steps:        generate, filter, evaluate, analyze
  🗂️  Stage logs: logs/design_pipeline_GFP_AutoPilot_GF

In [14]:
# Cell_8_Top10_Complex_Visualization_Viewer_with_PDF.py
# Requirement: Extract and render the top 10 generated designs from the Auto-Pilot master log. Apply a 3-tier coloring scheme: Target Base (Gray Mesh), Target Grooves (Orange Surface at explicit residues 39,40,41,73,74,200,201,202,203,204), and Binder (Bright Green/Lime). Output ONLY the Binder protein sequence without background highlighting. ADDITIONALLY, generate and save a comprehensive PDF report containing sequences and scores to Google Drive.

import os, json, csv, sys
from pathlib import Path
import py3Dmol
from IPython.display import display, HTML

# ---------------------------------------------------------
# 0. Dependency Resolution (Including PDF Engine)
# ---------------------------------------------------------
try:
    from fpdf import FPDF
except ImportError:
    print("📦 Installing missing PDF dependency (fpdf)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fpdf"])
    from fpdf import FPDF

# Standard amino acid 3-to-1 letter mapping dictionary
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# ---------------------------------------------------------
# 1. Configuration and Path Setup
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("Error: Global configuration file not found.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_chain = str(cfg.get('target_chains', 'A')).split(',')[0].strip()
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not master_log_path.exists() or not FIXED_PDB_PATH.exists():
    print("Error: Master log or template fixed.pdb not found. Pipeline cannot proceed.")
    sys.exit(1)

# ---------------------------------------------------------
# 2. Structural Subtraction Mapping (Extract baseline set)
# ---------------------------------------------------------
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

template_sel_list = []
for chain, resis in template_chains_resis.items():
    template_sel_list.append({'chain': chain, 'resi': list(resis)})

target_selection = {'or': template_sel_list}
binder_selection = {'not': target_selection}

# Define Target Grooves
groove_resis = ['39', '40', '41', '73', '74', '200', '201', '202', '203', '204']
groove_selection = {'chain': target_chain, 'resi': groove_resis}

display(HTML("<h2>--- Top 10 Generated Designs (Complex Visualization) ---</h2>"))

# ---------------------------------------------------------
# 3. Parse Log and Sort Top 10
# ---------------------------------------------------------
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['max_ipSAE'] = float(row['max_ipSAE'])
            data.append(row)
        except ValueError:
            pass

data.sort(key=lambda x: x['max_ipSAE'], reverse=True)
top_10 = data[:10]

if not top_10:
    print("Error: No valid entries found in the master log.")
    sys.exit(1)

# ---------------------------------------------------------
# 4. Initialize PDF Report Generator
# ---------------------------------------------------------
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()
pdf.set_font("Arial", style='B', size=14)
pdf.cell(200, 10, txt=f"Proteina-Complexa Generation Report", ln=True, align='C')
pdf.set_font("Arial", size=10)
pdf.cell(200, 10, txt=f"Task: {task_name} | Mode: Auto-Pilot Global Top 10", ln=True, align='C')
pdf.line(10, 30, 200, 30)
pdf.ln(10)

# ---------------------------------------------------------
# 5. Render Generated Visualizations and Write PDF
# ---------------------------------------------------------
for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['max_ipSAE']

    target_pdb = None
    if source_dir.exists():
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():
        display(HTML(f"<h3>Rank {rank}: {d_id} (ipSAE: {score:.4f})</h3>"))

        # Write header to PDF
        pdf.set_font("Arial", style='B', size=11)
        pdf.cell(0, 8, txt=f"Rank {rank}: {d_id} (Score: {score:.4f})", ln=True)

        print(f"   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...")

        with open(target_pdb, 'r') as f:
            pdb_data = f.read()

        # Parse PDB data to extract sequence
        chains_seq = {}
        for line in pdb_data.split('\n'):
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                res_name = line[17:20].strip()
                chain_id = line[21]
                if chain_id not in chains_seq:
                    chains_seq[chain_id] = ""
                chains_seq[chain_id] += d3to1.get(res_name, 'X')

        # Display extracted sequence for Binder only, and append to PDF
        pdf.set_font("Courier", size=9)
        for c, s in chains_seq.items():
            if c not in template_chains_resis.keys():
                # Output to Colab HTML
                display(HTML(f"<div style='font-family: monospace; margin-bottom: 10px;'><b>Binder Sequence (Chain {c}):</b> {s}</div>"))
                # Output to PDF
                pdf.multi_cell(0, 5, txt=f"Binder Sequence (Chain {c}): {s}")
        pdf.ln(5)

        # Execute 3D Rendering
        viewer = py3Dmol.view(width=800, height=500)
        viewer.addModel(pdb_data, 'pdb')

        # 1. Target Base: Gray cartoon and gray mesh (wireframe surface)
        viewer.setStyle(target_selection, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
        viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, target_selection)

        # 2. Target Grooves: Orange surface at explicit residues
        viewer.setStyle(groove_selection, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, groove_selection)

        # 3. Binder: Bright green (lime) cartoon and solid surface
        viewer.setStyle(binder_selection, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, binder_selection)

        viewer.zoomTo()
        viewer.show()
    else:
        display(HTML(f"<h3>Rank {rank}: {d_id} (ipSAE: {score:.4f}) - PDB File Not Found</h3>"))

# ---------------------------------------------------------
# 6. Finalize and Export PDF Report
# ---------------------------------------------------------
pdf_out_dir = BASE_DIR / 'screening_results' / task_name
pdf_out_dir.mkdir(parents=True, exist_ok=True)
pdf_out_path = pdf_out_dir / f"AutoPilot_Top10_Binder_Report.pdf"

pdf.output(str(pdf_out_path))
print(f"\n✅ PDF Output Complete: Document exported successfully.")
display(HTML(f"<div style='color: #155724; background-color: #d4edda; padding: 10px; border-radius: 5px; margin-top: 20px;'><b>📥 PDF Report saved to Google Drive:</b><br>{pdf_out_path.relative_to(BASE_DIR)}</div>"))

# ==============================================================================
# Update Log:
# Purpose: Extract and output only the Binder protein sequence dynamically from the PDB file, removing background HTML styling. Generate a downloadable PDF report logging the sequences and ipSAE scores utilizing the FPDF library.
# Upstream Code: Cell_8_Top10_Complex_Visualization_Viewer.py
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-08 08:05 EDT
# Changed Lines:
# - Lines 12-18: Integrated `fpdf` dependency resolution block.
# - Lines 94-102: Instantiated the `FPDF` object, formatting the global header.
# - Lines 134-136 & 151-158: Modified HTML output to only show Binder sequence without background color. Added PDF data insertion logic synchronized with the HTML generation.
# - Lines 179-186: Added final PDF output protocol to save `AutoPilot_Top10_Binder_Report.pdf` directly to Google Drive.
# ==============================================================================

   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

... (11 outputs left out)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


✅ PDF Output Complete: Document exported successfully.


In [ ]:

# =================================================================
# 💧 Cell_9_Top10_Solubility_Analyzer.py
# Requirement: Extract the Top 10 generated designs, dynamically identify
# the binder sequence using structural subtraction, and evaluate solubility.
# =================================================================

import os, json, csv
from pathlib import Path
try:
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
except ImportError:
    import subprocess
    print("installing Biopython")
    subprocess.check_call(["pip", "install", "biopython"])
    from Bio.SeqUtils.ProtParam import ProteinAnalysis

# 1. Configuration and Path Setup
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("Error: The global configuration file could not be located.")
    import sys; sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not master_log_path.exists() or not FIXED_PDB_PATH.exists():
    print("Error: The log file or the template file could not be located.")
    import sys; sys.exit(1)

# 2. --- [Build Target Exclusion Dictionary (for Extracting the Generated Peptide)] ---
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

# --- [Amino Acid Three-Letter to Single-Letter Code Mapping] ---
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# 3. Read Log file and extract Top 10
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['max_ipSAE'] = float(row['max_ipSAE'])
            data.append(row)
        except ValueError:
            pass

data.sort(key=lambda x: x['max_ipSAE'], reverse=True)
top_10 = data[:10]

if not top_10:
    print("rror: No valid data found in the log.")
    import sys; sys.exit(1)

print("\n🚀 --- [Top 10 Candidate Designs: Solubility and Physicochemical Properties Evaluation] ---")

for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['max_ipSAE']


    target_pdb = None
    if source_dir.exists():
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():

        binder_seq = ""
        with open(target_pdb, 'r') as f:
            for line in f:
                if line.startswith("ATOM") and line[12:16].strip() == "CA":
                    chain = line[21]
                    resi = line[22:26].strip()
                    resn = line[17:20].strip()

                    if chain not in template_chains_resis or resi not in template_chains_resis[chain]:
                        binder_seq += d3to1.get(resn, 'X')


        binder_seq = binder_seq.replace('X', '')

        if not binder_seq:
            print(f"\n[{rank}] {d_id} | Failed to retrieve binder sequence from the PDB file.")
            continue


        analysis = ProteinAnalysis(binder_seq)
        gravy_score = analysis.gravy()
        pi_value = analysis.isoelectric_point()
        charge_at_7_4 = analysis.charge_at_pH(7.4)


        print(f"\n🏅 Rank {rank} | Design: {d_id} | ipSAE: {score:.4f}")
        print(f"   🧬 Extracting sequence : {binder_seq}")

        # GRAVY Evaluation
        if gravy_score < 0:
            print(f"   💧 GRAVY score : {gravy_score:.3f} (Hydrophilic; typically exhibits good water solubility. ✅)")
        else:
            print(f"   🪨 GRAVY score : {gravy_score:.3f} (Hydrophobic; potential of aggregation and precipitation. ⚠️)")

        # pI Evaluation
        if 6.5 <= pi_value <= 8.5:
            print(f"   ⚡ pI : {pi_value:.2f} (⚠️ Possible protein precipitation)")
        else:
            print(f"   ⚡ pI : {pi_value:.2f} (Good solubility ✅)")

        print(f"   🔋 Net charge under physiological conditions (pH 7.4) : {charge_at_7_4:.2f} (Higher absolute value: Stronger electrostatic repulsion, lower risk of aggregation.)")

    else:
        print(f"\n🏅 Rank {rank} | {d_id} | ❌ PDB file not found; analysis sequence aborted.")

print("\n" + "="*60)
print("📌 Interpretation:")
print("- GRAVY < 0 and pI away from 7.4 has higher solubility。")
print("- High binding score with high GRAVY: Recommend subsequent hydrophilic mutations on non-binding interfaces to mitigate aggregation chance.")


🚀 --- [Top 10 Candidate Designs: Solubility and Physicochemical Properties Evaluation] ---

🏅 Rank 1 | Design: Design_6 | ipSAE: 0.7920
   🧬 Extracting sequence : VITTTAHALPGT
   🪨 GRAVY score : 0.675 (Hydrophobic; potential of aggregation and precipitation. ⚠️)
   ⚡ pI : 6.71 (⚠️ Possible protein precipitation)
   🔋 Net charge under physiological conditions (pH 7.4) : -0.44 (Higher absolute value: Stronger electrostatic repulsion, lower risk of aggregation.)

🏅 Rank 2 | Design: Design_2 | ipSAE: 0.7911
   🧬 Extracting sequence : VITTTAHALPGT
   🪨 GRAVY score : 0.675 (Hydrophobic; potential of aggregation and precipitation. ⚠️)
   ⚡ pI : 6.71 (⚠️ Possible protein precipitation)
   🔋 Net charge under physiological conditions (pH 7.4) : -0.44 (Higher absolute value: Stronger electrostatic repulsion, lower risk of aggregation.)

🏅 Rank 3 | Design: Design_5 | ipSAE: 0.7898
   🧬 Extracting sequence : TITTTAHAIK
   🪨 GRAVY score : 0.270 (Hydrophobic; potential of aggregation and precipitati

In [ ]:
# Cell_10_Auto_Download_Top_10.py
# Requirement: Automatically extract the top 10 generated designs from the most recent Auto-Pilot master log, package their PDB files, and trigger a browser download without hardcoding filenames.
##################################################
################ download top 10 files############
##################################################
import os, json, csv, shutil
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    print("⚠️ Warning: Not running in Google Colab environment. Files will be packaged but browser download will not trigger.")

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Pipeline cannot proceed.")
    import sys; sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'

if not master_log_path.exists():
    print(f"❌ Error: Master log not found at {master_log_path.relative_to(BASE_DIR)}.")
    print("Please ensure the Auto-Pilot Generation Loop has successfully recorded at least one cycle.")
    import sys; sys.exit(1)

print("🔍 Parsing global screening log for top candidates...")

# Read and sanitize the CSV log data
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['max_ipSAE'] = float(row['max_ipSAE'])
            data.append(row)
        except ValueError:
            pass

if not data:
    print("❌ Error: No valid data entries found in the master log.")
    import sys; sys.exit(1)

# Sort descending by max_ipSAE and slice top 10
data.sort(key=lambda x: x['max_ipSAE'], reverse=True)
top_10 = data[:10]

# Establish a temporary staging directory
STAGING_DIR = BASE_DIR / f'Top_10_{task_name}_Staging'
STAGING_DIR.mkdir(parents=True, exist_ok=True)

copied_count = 0
for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['max_ipSAE']

    target_pdb = None
    if source_dir.exists():
        # Iterate over all PDBs in the source inference directory to locate the matching structure
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break

        # Fallback metric if exact ID is malformed
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():
        new_name = f"Rank{rank}_{d_id}_score{score:.4f}.pdb"
        shutil.copy(target_pdb, STAGING_DIR / new_name)
        copied_count += 1
        print(f"✅ Packaged Rank {rank}: {new_name}")
    else:
        print(f"⚠️ Could not locate corresponding PDB file for {d_id} in {source_dir.relative_to(BASE_DIR)}")

if copied_count > 0:
    ZIP_PATH = BASE_DIR / 'screening_results' / task_name / f'Top_10_AutoPilot_{task_name}'
    print("\n📦 Compressing top candidates into a ZIP archive...")
    shutil.make_archive(str(ZIP_PATH), 'zip', STAGING_DIR)
    final_zip = str(ZIP_PATH) + '.zip'

    # Clean up the staging directory
    shutil.rmtree(STAGING_DIR)

    print("⬇️ Triggering browser download. Please monitor for download prompts...")
    try:
        files.download(final_zip)
    except NameError:
        pass
else:
    print("\n❌ Failed to copy any valid PDB files. Archive generation aborted.")
    shutil.rmtree(STAGING_DIR)

# Purpose: Automate the extraction, packaging, and downloading of the top 10 generated PDB structures based on the auto-pilot log output.
# Upstream Code: Auto-Pilot Engine (Cell_9b).
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 09:37 EDT.
# Changed Lines:
# * This is a completely new cell logic designed to execute independently after Cell 9b completes.

🔍 Parsing global screening log for top candidates...
✅ Packaged Rank 1: Rank1_Design_6_score0.7920.pdb
✅ Packaged Rank 2: Rank2_Design_2_score0.7911.pdb
✅ Packaged Rank 3: Rank3_Design_5_score0.7898.pdb
✅ Packaged Rank 4: Rank4_Design_1_score0.7898.pdb
✅ Packaged Rank 5: Rank5_Design_1_score0.7897.pdb
✅ Packaged Rank 6: Rank6_Design_2_score0.7845.pdb
✅ Packaged Rank 7: Rank7_Design_4_score0.7837.pdb
✅ Packaged Rank 8: Rank8_Design_3_score0.7807.pdb
✅ Packaged Rank 9: Rank9_Design_3_score0.7771.pdb
✅ Packaged Rank 10: Rank10_Design_7_score0.7737.pdb

📦 Compressing top candidates into a ZIP archive...
⬇️ Triggering browser download. Please monitor for download prompts...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Section 5: Surface Scanning & Hotspot Targeted Screening

In [27]:

# Cell_15_Geometric_Surface_Patch_Scanning_Engine.py
# Requirement: Determine exposed surface residues based on anchored geometrical boundaries and sequentially process designs for derived hotspot patches.
# Implement a robust checkpoint/resume mechanism (断点续传) to survive Colab disconnections. Inject dynamic random seeds per patch to guarantee stochastic structural diversity.

import os, json, sys, subprocess, yaml, textwrap, shutil, datetime, time, random
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.spatial import KDTree, Delaunay

print("🔍 Initializing Geometric Surface Scanning Engine...")

# ---------------------------------------------------------
# 1. Load Persistent Configuration
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please navigate to Section 0 and configure parameters.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_chains = cfg['target_chains']
anchor_str = str(cfg['anchor_residues'])
scan_margin = float(cfg['scan_margin'])
local_patch_radius = float(cfg['local_patch_radius'])
designs_per_patch = int(cfg['designs_per_patch'])
batch_size = int(cfg['scan_batch_size'])

anchor_res_list = [int(x.strip()) for x in anchor_str.split(',') if x.strip()]

SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not SOURCE_PDB_PATH.exists():
    print(f"❌ Error: Required processed PDB structure not found at {SOURCE_PDB_PATH}. Prior execution of Section 2 is mandatory.")
    sys.exit(1)

SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"GeoScan_{task_name}_{SESSION_TIMESTAMP}"
SESSION_MASTER_DIR = BASE_DIR / 'assets' / 'target_data' / SESSION_NAME
SESSION_MASTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Master Scan Directory established at: {SESSION_MASTER_DIR.relative_to(BASE_DIR)}")

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m}m {s}s"

# ---------------------------------------------------------
# 2. Parse PDB for Spatial Coordinates & Amino Acid Nomenclature
# ---------------------------------------------------------
coords, res_ids, res_names = [], [], []

with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
            res_names.append(line[17:20].strip())
            res_ids.append(int(line[22:26].strip()))
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

coords, res_ids, res_names = np.array(coords), np.array(res_ids), np.array(res_names)
protein_kdtree = KDTree(coords)

SURFACE_NEIGHBOR_THRESHOLD = 24

# ---------------------------------------------------------
# 3. Anchor Validation & Geometric Convex Hull Construction
# ---------------------------------------------------------
anchor_coords = []
for a_res in anchor_res_list:
    if a_res in res_ids:
        idx = np.where(res_ids == a_res)[0][0]
        neighbors_10A = protein_kdtree.query_ball_point(coords[idx], r=10.0)
        if len(neighbors_10A) > SURFACE_NEIGHBOR_THRESHOLD:
            sys.exit(f"\n❌ FATAL ERROR: Specified Anchor {a_res} ({res_names[idx]}) is mathematically buried within the protein core!")
        anchor_coords.append(coords[idx])
    else:
        sys.exit(f"\n❌ FATAL ERROR: Requested Anchor Residue {a_res} cannot be mapped to target chain {target_chains}.")

if not anchor_coords:
    sys.exit("\n❌ Error: Geometry construction aborted. No validated anchor residues acquired.")

anchor_coords = np.array(anchor_coords)
shape_points = []
spacing = 1.0

n_anchors = len(anchor_coords)
if n_anchors == 1:
    shape_points = anchor_coords
elif n_anchors == 2:
    d = np.linalg.norm(anchor_coords[1] - anchor_coords[0])
    shape_points = np.linspace(anchor_coords[0], anchor_coords[1], max(2, int(d / spacing)))
elif n_anchors == 3:
    d1, d2 = np.linalg.norm(anchor_coords[1] - anchor_coords[0]), np.linalg.norm(anchor_coords[2] - anchor_coords[0])
    for u in np.linspace(0, 1, max(2, int(d1/spacing))):
        for v in np.linspace(0, 1-u, max(2, int(d2/spacing))):
            shape_points.append(anchor_coords[0] + u*(anchor_coords[1]-anchor_coords[0]) + v*(anchor_coords[2]-anchor_coords[0]))
    shape_points = np.array(shape_points)
else:
    hull = Delaunay(anchor_coords, qhull_options='QJ')
    min_b, max_b = np.min(anchor_coords, axis=0), np.max(anchor_coords, axis=0)
    grid_x, grid_y, grid_z = np.mgrid[min_b[0]:max_b[0]:spacing, min_b[1]:max_b[1]:spacing, min_b[2]:max_b[2]:spacing]
    grid_pts = np.vstack((grid_x.ravel(), grid_y.ravel(), grid_z.ravel())).T
    inside = hull.find_simplex(grid_pts) >= 0
    shape_points = grid_pts[inside]

shape_kdtree = KDTree(shape_points if shape_points.ndim == 2 else [shape_points])
candidate_indices = shape_kdtree.query_ball_tree(protein_kdtree, r=scan_margin)
candidate_idx_set = set([idx for sublist in candidate_indices for idx in sublist])

# ---------------------------------------------------------
# 4. Rigorous Topographic Filtration & Resume State Interception (断点续传拦截)
# ---------------------------------------------------------
surface_residues_raw = []
for idx in candidate_idx_set:
    neighbors_10A = protein_kdtree.query_ball_point(coords[idx], r=10.0)
    if len(neighbors_10A) <= SURFACE_NEIGHBOR_THRESHOLD:
        surface_residues_raw.append(idx)

surface_residues = sorted(surface_residues_raw, key=lambda idx: res_ids[idx])

# Define persistent checkpoint file
CHECKPOINT_CSV = BASE_DIR / 'screening_results' / task_name / f'{task_name}_geoscan_checkpoint.csv'
CHECKPOINT_CSV.parent.mkdir(parents=True, exist_ok=True)

all_scan_results = []
completed_res_ids = set()

if CHECKPOINT_CSV.exists():
    try:
        df_checkpoint = pd.read_csv(CHECKPOINT_CSV)
        if not df_checkpoint.empty:
            all_scan_results = df_checkpoint.to_dict('records')
            for val in df_checkpoint['Center_Residue']:
                # Parse format "8 (PHE)" to extract integer 8
                res_id_str = str(val).split('(')[0].strip()
                if res_id_str.isdigit():
                    completed_res_ids.add(int(res_id_str))
            print(f"📥 RESUME STATE DETECTED: Successfully loaded {len(completed_res_ids)} previously processed patch geometries from checkpoint.")
    except Exception as e:
        print(f"⚠️ Warning: Checkpoint file exists but could not be parsed: {e}. Starting fresh.")

# Filter out already completed residues to prevent redundant GPU execution
pending_surface_residues = [idx for idx in surface_residues if res_ids[idx] not in completed_res_ids]

# ---------------------------------------------------------
# 5. Detection Log Formatting
# ---------------------------------------------------------
print("\n" + "="*70)
print(f"🎯 ALGORITHMIC GEOMETRIC SCANNING SECTOR ESTABLISHED")
print(f"Geometry Parameters: {n_anchors} Fixed Anchors | Volumetric Expansion Margin: {scan_margin}Å")
print(f"Total Isolated Candidates: {len(surface_residues)} residues.")
print(f"Pending Execution Queue  : {len(pending_surface_residues)} residues remaining.")
print("-" * 70)

res_strings = [f"{res_ids[idx]}({res_names[idx]})" for idx in pending_surface_residues]
formatted_list = textwrap.fill(", ".join(res_strings), width=80, initial_indent="➤ ", subsequent_indent="  ")
print(formatted_list if res_strings else "➤ Queue Empty. All patches have been scanned.")
print("="*70)

if not pending_surface_residues:
    print("✅ Geometric scanning fully completed based on checkpoint data. Proceeding to final analytics.")
else:
    print("🚀 Queuing generative algorithmic pipelines targeting the pending topographical patches...\n")

# ---------------------------------------------------------
# 6. Core Execution Loop: Patch Isolation & Candidate Synthesis
# ---------------------------------------------------------
env = os.environ.copy()
env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

for i, idx in enumerate(pending_surface_residues, 1):
    loop_start_time = time.time()
    center_res = res_ids[idx]
    center_name = res_names[idx]

    local_neighbors = protein_kdtree.query_ball_point(coords[idx], r=local_patch_radius)
    patch_residues = sorted([res_ids[n] for n in local_neighbors])
    patch_str = ",".join(map(str, patch_residues))

    patch_run_name = f"{SESSION_NAME}_Res{center_res}"
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / patch_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 [Active Sweep {i}/{len(pending_surface_residues)}] Center Assignment: {center_res} ({center_name})")
    print(f"📂 Local Sandbox: {RUN_ISOLATED_DIR.relative_to(SESSION_MASTER_DIR)}")
    print(f"=======================================================")

    yaml_path = BASE_DIR / 'configs/targets/targets_dict.yaml'
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)

    yaml_data['target_dict_cfg'][task_name]['hotspot_residues'] = [f"{target_chains}{r}" for r in patch_residues]

    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    # Dynamic Random Seed Injection
    dynamic_seed = random.randint(1, 999999)

    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={patch_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={designs_per_patch} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.seed={dynamic_seed} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=False ++run_analyze=False"
    )

    process = subprocess.run(f"source env.sh && {cmd_str}", env=env, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), check=False)
    ret_code = process.returncode

    inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{patch_run_name}'

    if ret_code == 0:
        valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

        if valid_csvs:
            target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
            try:
                df = pd.read_csv(target_csv)
                score_col = 'af2folding_max_ipsae' if 'af2folding_max_ipsae' in df.columns else 'total_reward'

                if score_col in df.columns:
                    df_sorted = df.sort_values(by=score_col, ascending=False)
                    top_score = df_sorted[score_col].max()

                    print(f"📝 Evaluation Metrics for Patch {center_res} (Ranked by {score_col}):")
                    print(f"  Rank | Design_ID    | Score")
                    print("  " + "-"*35)

                    for rank, (csv_index, row) in enumerate(df_sorted.iterrows(), 1):
                        d_id = str(row.get('design_id', '')).strip()
                        if not d_id or str(d_id).lower() == 'nan':
                            d_id = f"Design_{row.get('pdb_index', rank)}"
                        s_val = float(row.get(score_col, -999.0))
                        print(f"  {rank:<4} | {d_id[:12]:<12} | {s_val:.4f}")

                    result_entry = {
                        'Center_Residue': f"{center_res} ({center_name})",
                        f'Max_{score_col}': round(top_score, 4),
                        'Designs_Generated': len(df_sorted),
                        'Hotspot_Patch': patch_str
                    }
                    all_scan_results.append(result_entry)

                    # Real-time Checkpointing (实时落盘)
                    pd.DataFrame(all_scan_results).to_csv(CHECKPOINT_CSV, index=False)
                    print(f"💾 Checkpoint Updated: Scan metrics for {center_res} securely written to disk.")

            except Exception as e:
                print(f"⚠️ Data Sync Failure: Unable to parse batch pipeline arrays for {center_res}: {e}")
        else:
            print(f"⚠️ Warning: No valid CSV metrics output located in {inference_dir.name}.")
    else:
        print(f"⚠️ Alert: Generative pipeline exited with non-zero status code {ret_code}. Skipping metric ingestion for {center_res}.")

    cycle_duration = format_time(time.time() - loop_start_time)
    print(f"\n⏱️ Patch Sweep {i} Completed in: {cycle_duration}")

    if i < len(pending_surface_residues):
        import gc
        gc.collect()
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        print("🧹 Buffer memory cleared. Initiating next spatial coordinate...")
        time.sleep(2)

# ---------------------------------------------------------
# 7. Global Analytics and Final Post-Processing Report Output
# ---------------------------------------------------------
if all_scan_results:
    results_df = pd.DataFrame(all_scan_results)

    sort_col = [col for col in results_df.columns if col.startswith('Max_')][0]
    results_df = results_df.sort_values(by=sort_col, ascending=False)

    FINAL_CSV = BASE_DIR / 'screening_results' / task_name / f'geometric_scan_results_FINAL_{SESSION_TIMESTAMP}.csv'
    results_df.to_csv(FINAL_CSV, index=False)

    # Clean up checkpoint after successful completion
    if CHECKPOINT_CSV.exists():
        os.remove(CHECKPOINT_CSV)

    print("\n🏆 ======================================================= 🏆")
    print("           GEOMETRIC SCANNING PROTOCOL CONCLUDED              ")
    print("🏆 ======================================================= 🏆")
    print(f"Top 10 Prioritized Coordinates (Ranked by Peak Binding Scores):")
    print(results_df[['Center_Residue', sort_col, 'Designs_Generated']].head(10).to_string(index=False))
    print(f"\n📂 Comprehensive analytic report finalized and committed to: {FINAL_CSV}")
else:
    print("\n🛑 Geometric Scanning Protocol Terminated. No viable candidates were successfully generated or evaluated across any surface patches.")

# ==============================================================================
# Update Log:
# Purpose: Implement stateful checkpointing to enable robust resume capabilities after Colab session timeouts. Inject dynamic random seeds to override framework defaults and ensure stochastic generation diversity per patch.
# Upstream Code: Original Cell 10 Geometric Surface Patch Scanning Engine.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 18:46 EDT
# Changed Lines:
# - Line 7: Imported `random` library.
# - Lines 129-145: Engineered the checkpoint interception logic. The engine parses the static `CHECKPOINT_CSV`, extracts integer values from the `Center_Residue` column, and populates `completed_res_ids`.
# - Lines 147-148: Created the `pending_surface_residues` queue, dynamically subtracting already processed spatial patches to bypass redundant execution.
# - Lines 191-192, 196: Injected `dynamic_seed = random.randint(1, 999999)` and embedded `++generation.seed={dynamic_seed}` into the Hydra command line to enforce physical conformation diversity.
# - Lines 231-233: Executed real-time IO disk writes. Appends data to `CHECKPOINT_CSV` immediately after each successful patch evaluation loop.
# - Lines 259-264: Refactored terminal block. Upon full job completion, the engine exports data to a timestamped `FINAL_CSV` and gracefully deletes the temporary `CHECKPOINT_CSV` state file.
# ==============================================================================

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [ ]:
# Cell_16_Complex_Visualization_Viewer_with_PDF.py
# Requirement: Automatically locate the most recent inference session from the `inference` directory. Extract top 2 designs per site. Render the 3D structure using py3Dmol with a 3-tier coloring scheme. Output ONLY the Binder protein sequence. ADDITIONALLY, generate and save a comprehensive PDF report containing all sequences and scores to Google Drive.

import os, json, sys, re
from pathlib import Path
import pandas as pd
import py3Dmol
from IPython.display import display, HTML

# ---------------------------------------------------------
# 0. Dependency Resolution (Including PDF Engine)
# ---------------------------------------------------------
try:
    from fpdf import FPDF
except ImportError:
    print("📦 Installing missing PDF dependency (fpdf)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fpdf"])
    from fpdf import FPDF

# Standard amino acid 3-to-1 letter mapping dictionary
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# ---------------------------------------------------------
# 1. Configuration and Path Setup
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("Error: Global configuration file not found.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
# Extract primary target chain for groove mapping
target_chain = str(cfg.get('target_chains', 'A')).split(',')[0].strip()

FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not FIXED_PDB_PATH.exists():
    print("Error: Template fixed.pdb not found. Pipeline cannot proceed.")
    sys.exit(1)

# ---------------------------------------------------------
# 2. Structural Subtraction Mapping (Extract baseline set)
# ---------------------------------------------------------
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

template_sel_list = []
for chain, resis in template_chains_resis.items():
    template_sel_list.append({'chain': chain, 'resi': list(resis)})

target_selection = {'or': template_sel_list}
binder_selection = {'not': target_selection}

# Defined target grooves based on explicit requirement
groove_resis = ['39', '40', '41', '73', '74', '200', '201', '202', '203', '204']
groove_selection = {'chain': target_chain, 'resi': groove_resis}

# ---------------------------------------------------------
# 3. Dynamically Locate Latest Inference Session
# ---------------------------------------------------------
inference_dir = BASE_DIR / 'inference'
if not inference_dir.exists():
    print(f"Error: Inference directory {inference_dir} does not exist.")
    sys.exit(1)

all_task_inferences = [d for d in inference_dir.glob(f"search_binder_local_pipeline_{task_name}_*") if d.is_dir()]

if not all_task_inferences:
    print(f"Error: No inference runs found for task {task_name}.")
    sys.exit(1)

latest_dir = max(all_task_inferences, key=lambda x: x.stat().st_mtime)

def get_session_name(d_name):
    return re.sub(r'(_Res\d+|_Cycle\d+|_Patch\d+)$', '', d_name)

latest_session_prefix = get_session_name(latest_dir.name)
session_dirs = [d for d in all_task_inferences if d.name.startswith(latest_session_prefix)]

print(f"🔍 Dynamic Data Ingestion Active.")
print(f"📂 Detected Latest Session: {latest_session_prefix}")
print(f"📊 Analyzing {len(session_dirs)} output directories...\n")

display(HTML("<h2>--- Top 2 Generated Designs Per Site (Complex Visualization) ---</h2>"))

# ---------------------------------------------------------
# 4. Initialize PDF Report Generator
# ---------------------------------------------------------
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()
pdf.set_font("Arial", style='B', size=14)
pdf.cell(200, 10, txt=f"Proteina-Complexa Generation Report", ln=True, align='C')
pdf.set_font("Arial", size=10)
pdf.cell(200, 10, txt=f"Task: {task_name} | Session: {latest_session_prefix}", ln=True, align='C')
pdf.line(10, 30, 200, 30)
pdf.ln(10)

# ---------------------------------------------------------
# 5. Parse CSVs and Extract Top 2 Per Site
# ---------------------------------------------------------
top_designs_per_site = []

for s_dir in session_dirs:
    match = re.search(r'(_Res\d+|_Cycle\d+|_Patch\d+)$', s_dir.name)
    site_label = match.group(1).lstrip('_') if match else "Global"

    valid_csvs = [f for f in s_dir.rglob('*.csv') if 'timing' not in f.name.lower()]
    if not valid_csvs:
        continue

    target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)

    try:
        df = pd.read_csv(target_csv)
        score_col = 'af2folding_max_ipsae' if 'af2folding_max_ipsae' in df.columns else ('max_ipSAE' if 'max_ipSAE' in df.columns else 'total_reward')

        if score_col in df.columns:
            df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
            df_sorted = df.dropna(subset=[score_col]).sort_values(by=score_col, ascending=False).head(2)

            for rank, (_, row) in enumerate(df_sorted.iterrows(), 1):
                d_id = str(row.get('design_id', '')).strip()
                if not d_id or d_id.lower() == 'nan':
                    d_id = f"Design_{row.get('pdb_index', rank)}"

                pdb_path_str = str(row.get('pdb_path', '')).strip()

                entry = {
                    'site': site_label,
                    'site_rank': rank,
                    'design_id': d_id,
                    'score': float(row[score_col]),
                    'source_dir': s_dir,
                    'pdb_path_str': pdb_path_str,
                    'score_col_name': score_col
                }
                top_designs_per_site.append(entry)
    except Exception as e:
        print(f"Warning: Could not parse metrics for {site_label} from {target_csv.name}: {e}")

if not top_designs_per_site:
    print("Error: No valid metrics could be extracted from the active session.")
    sys.exit(1)

def natural_site_sort_key(item):
    site = item['site']
    match = re.match(r'([A-Za-z]+)(\d+)', site)
    if match:
        return (match.group(1), int(match.group(2)), item['site_rank'])
    return (site, 0, item['site_rank'])

top_designs_per_site.sort(key=natural_site_sort_key)

# ---------------------------------------------------------
# 6. Render Visualizations, Print Sequences, and Write to PDF
# ---------------------------------------------------------
for entry in top_designs_per_site:
    source_dir = entry['source_dir']
    d_id = entry['design_id']
    score = entry['score']
    site = entry['site']
    rank = entry['site_rank']
    score_col = entry['score_col_name']
    pdb_path_str = entry['pdb_path_str']

    target_pdb = None

    if pdb_path_str and pdb_path_str.lower() != 'nan':
        possible_pdb = Path(pdb_path_str)
        if possible_pdb.is_file():
            target_pdb = possible_pdb
        else:
            possible_pdb_rel = source_dir / possible_pdb.name
            if possible_pdb_rel.is_file():
                target_pdb = possible_pdb_rel

    if not target_pdb:
        print(f"\n❌ ERROR: Strict PDB path validation failed for Design '{d_id}' at Site '{site}'.")
        print(f"Recorded CSV path is invalid or missing: {pdb_path_str}")
        print("Pipeline aborted due to structural data inconsistency.")
        sys.exit(1)

    display(HTML(f"<h3>Site: {site} | Rank {rank}: {d_id} ({score_col}: {score:.4f})</h3>"))

    # Write header to PDF
    pdf.set_font("Arial", style='B', size=11)
    pdf.cell(0, 8, txt=f"Site: {site} | Rank {rank}: {d_id} (Score: {score:.4f})", ln=True)

    print(f"   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...")

    with open(target_pdb, 'r') as f:
        pdb_data = f.read()

    # Parse PDB data to extract sequence
    chains_seq = {}
    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            res_name = line[17:20].strip()
            chain_id = line[21]
            if chain_id not in chains_seq:
                chains_seq[chain_id] = ""
            chains_seq[chain_id] += d3to1.get(res_name, 'X')

    # Display extracted sequence for Binder only, and append to PDF
    pdf.set_font("Courier", size=9)
    for c, s in chains_seq.items():
        if c not in template_chains_resis.keys():
            # Output to Colab HTML
            display(HTML(f"<div style='font-family: monospace; margin-bottom: 10px;'><b>Binder Sequence (Chain {c}):</b> {s}</div>"))
            # Output to PDF
            pdf.multi_cell(0, 5, txt=f"Binder Sequence (Chain {c}): {s}")
    pdf.ln(5)

    # Execute 3D Rendering
    viewer = py3Dmol.view(width=800, height=500)
    viewer.addModel(pdb_data, 'pdb')

    # 1. Target Base: Gray cartoon and gray mesh (wireframe surface)
    viewer.setStyle(target_selection, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, target_selection)

    # 2. Target Grooves: Orange surface at explicit residues
    viewer.setStyle(groove_selection, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
    viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, groove_selection)

    # 3. Binder: Bright green (lime) cartoon and solid surface
    viewer.setStyle(binder_selection, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
    viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, binder_selection)

    viewer.zoomTo()
    viewer.show()

# ---------------------------------------------------------
# 7. Finalize and Export PDF Report
# ---------------------------------------------------------
pdf_out_dir = BASE_DIR / 'screening_results' / task_name
pdf_out_dir.mkdir(parents=True, exist_ok=True)
pdf_out_path = pdf_out_dir / f"{latest_session_prefix}_Binder_Report.pdf"

pdf.output(str(pdf_out_path))
print(f"\n✅ PDF Output Complete: Document exported successfully.")
display(HTML(f"<div style='color: #155724; background-color: #d4edda; padding: 10px; border-radius: 5px; margin-top: 20px;'><b>📥 PDF Report saved to Google Drive:</b><br>{pdf_out_path.relative_to(BASE_DIR)}</div>"))

# ==============================================================================
# Update Log:
# Purpose: Maintain the 3-tier visual hierarchy rendering and Binder sequence output, whilst simultaneously generating a downloadable, highly structured PDF report logging the generated sequences and ipSAE scores utilizing the FPDF library.
# Upstream Code: Cell_16_Complex_Visualization_Viewer.py
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-08 07:52 EDT
# Changed Lines:
# - Lines 14-20: Integrated `fpdf` dependency resolution block for runtime installation.
# - Lines 103-111: Instantiated the `FPDF` object, formatting the global header with task and session metadata.
# - Lines 218-220 & 237-244: Synchronized Colab terminal execution with `fpdf.cell` and `fpdf.multi_cell` writes. The sequence layout strictly utilizes monospace Courier font for sequence alignment in the document.
# - Lines 262-269: Added termination logic to save the compiled PDF directly to the active Google Drive screening directory (`screening_results`).
# ==============================================================================

In [ ]:
# Cell_17_Auto_Download_GeoScan_Top_Candidates.py
# Requirement: Automatically locate the most recent GeoScan inference session, extract the top 2 generated designs per spatial patch, package their PDB files with descriptive names, and trigger a browser download.

import os, json, sys, re, shutil
from pathlib import Path
import pandas as pd

try:
    from google.colab import files
except ImportError:
    print("⚠️ Warning: Not running in Google Colab environment. Files will be packaged but browser download will not trigger.")

# ---------------------------------------------------------
# 1. Configuration and Path Setup
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Pipeline cannot proceed.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']

# ---------------------------------------------------------
# 2. Dynamically Locate Latest Inference Session
# ---------------------------------------------------------
inference_dir = BASE_DIR / 'inference'
if not inference_dir.exists():
    print(f"❌ Error: Inference directory {inference_dir} does not exist.")
    sys.exit(1)

all_task_inferences = [d for d in inference_dir.glob(f"search_binder_local_pipeline_{task_name}_*") if d.is_dir()]

if not all_task_inferences:
    print(f"❌ Error: No inference runs found for task {task_name}.")
    sys.exit(1)

latest_dir = max(all_task_inferences, key=lambda x: x.stat().st_mtime)

def get_session_name(d_name):
    return re.sub(r'(_Res\d+|_Cycle\d+|_Patch\d+)$', '', d_name)

latest_session_prefix = get_session_name(latest_dir.name)
session_dirs = [d for d in all_task_inferences if d.name.startswith(latest_session_prefix)]

print(f"🔍 Parsing GeoScan directories for top candidates...")
print(f"📂 Detected Latest Session: {latest_session_prefix}")

# ---------------------------------------------------------
# 3. Parse CSVs and Extract Top 2 Per Site
# ---------------------------------------------------------
top_designs_per_site = []

for s_dir in session_dirs:
    match = re.search(r'(_Res\d+|_Cycle\d+|_Patch\d+)$', s_dir.name)
    site_label = match.group(1).lstrip('_') if match else "Global"

    valid_csvs = [f for f in s_dir.rglob('*.csv') if 'timing' not in f.name.lower()]
    if not valid_csvs:
        continue

    target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)

    try:
        df = pd.read_csv(target_csv)
        score_col = 'af2folding_max_ipsae' if 'af2folding_max_ipsae' in df.columns else ('max_ipSAE' if 'max_ipSAE' in df.columns else 'total_reward')

        if score_col in df.columns:
            df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
            df_sorted = df.dropna(subset=[score_col]).sort_values(by=score_col, ascending=False).head(2)

            for rank, (_, row) in enumerate(df_sorted.iterrows(), 1):
                d_id = str(row.get('design_id', '')).strip()
                if not d_id or d_id.lower() == 'nan':
                    d_id = f"Design_{row.get('pdb_index', rank)}"

                pdb_path_str = str(row.get('pdb_path', '')).strip()

                entry = {
                    'site': site_label,
                    'site_rank': rank,
                    'design_id': d_id,
                    'score': float(row[score_col]),
                    'source_dir': s_dir,
                    'pdb_path_str': pdb_path_str,
                    'score_col_name': score_col
                }
                top_designs_per_site.append(entry)
    except Exception as e:
        print(f"⚠️ Warning: Could not parse metrics for {site_label} from {target_csv.name}: {e}")

if not top_designs_per_site:
    print("❌ Error: No valid metrics could be extracted from the active session.")
    sys.exit(1)

def natural_site_sort_key(item):
    site = item['site']
    match = re.match(r'([A-Za-z]+)(\d+)', site)
    if match:
        return (match.group(1), int(match.group(2)), item['site_rank'])
    return (site, 0, item['site_rank'])

top_designs_per_site.sort(key=natural_site_sort_key)

# ---------------------------------------------------------
# 4. Package Files to Staging Directory
# ---------------------------------------------------------
STAGING_DIR = BASE_DIR / f'Top_GeoScan_{task_name}_Staging'
STAGING_DIR.mkdir(parents=True, exist_ok=True)

copied_count = 0
for entry in top_designs_per_site:
    source_dir = entry['source_dir']
    d_id = entry['design_id']
    score = entry['score']
    site = entry['site']
    rank = entry['site_rank']
    pdb_path_str = entry['pdb_path_str']

    target_pdb = None

    # Resolve physical path
    if pdb_path_str and pdb_path_str.lower() != 'nan':
        possible_pdb = Path(pdb_path_str)
        if possible_pdb.is_file():
            target_pdb = possible_pdb
        else:
            possible_pdb_rel = source_dir / possible_pdb.name
            if possible_pdb_rel.is_file():
                target_pdb = possible_pdb_rel

    if target_pdb and target_pdb.exists():
        new_name = f"Site_{site}_Rank{rank}_{d_id}_score{score:.4f}.pdb"
        shutil.copy(target_pdb, STAGING_DIR / new_name)
        copied_count += 1
        print(f"✅ Packaged: {new_name}")
    else:
        print(f"⚠️ Missing PDB: Could not locate structure for Site {site} Rank {rank} ({d_id})")

# ---------------------------------------------------------
# 5. Compress and Download
# ---------------------------------------------------------
if copied_count > 0:
    ZIP_PATH = BASE_DIR / 'screening_results' / task_name / f'Top_Candidates_{latest_session_prefix}'
    ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

    print("\n📦 Compressing candidates into a ZIP archive...")
    shutil.make_archive(str(ZIP_PATH), 'zip', STAGING_DIR)
    final_zip = str(ZIP_PATH) + '.zip'

    # Clean up the staging directory
    shutil.rmtree(STAGING_DIR)

    print("⬇️ Triggering browser download. Please monitor for download prompts...")
    try:
        files.download(final_zip)
    except NameError:
        pass
else:
    print("\n❌ Failed to copy any valid PDB files. Archive generation aborted.")
    shutil.rmtree(STAGING_DIR)

# ==============================================================================
# Update Log:
# Purpose: Automate the extraction, renaming, packaging, and downloading of the top 2 generated PDB structures per spatial patch derived from the GeoScan pipeline.
# Upstream Code: GeoScan Engine and Visualization Viewer (Cell 15/16).
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-08 08:00 EDT.
# Changed Lines:
# - Replaced single master log parsing with robust multi-directory recursive CSV scanning matched to Cell 16 logic.
# - Implemented structured renaming output (`Site_{site}_Rank{rank}_{d_id}_score{score:.4f}.pdb`) to clearly identify the geometric origin of each file in the downloaded archive.
# ==============================================================================

# ⚠️ CRITICAL: Irreversible Environment Eradication & Total Data Purge. Back up Data Before Execution.

In [ ]:
# # Cell_16_Forced_cleanup.py


# # ==========================================
# # Forced Environment Cleanup and State Synchronization (Python Native + Terminal Force)
# # ==========================================
# import os
# import shutil

# ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
# BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"

# print(">>> Initiating full-path forced cleanup sequence...")

# # 1. Use Terminal commands for physical erasure (handling FUSE mount synchronization)
# print(">>> [1/2] Forcing directory deletion via terminal commands...")
# !rm -rf "{ROOT_DIR}"
# !rm -rf "{BACKUP_DIR}"

# # 2. Use native Python libraries for kernel-state verification
# print(">>> [2/2] Verifying kernel-state via Python...")
# for target_path in [ROOT_DIR, BACKUP_DIR]:
#     if os.path.exists(target_path):
#         print(f"⚠️ Warning: Remnants detected after terminal command. Initiating secondary destruction: {target_path}")
#         try:
#             shutil.rmtree(target_path)
#             print(f"✅ Completely wiped: {target_path}")
#         except Exception as e:
#             print(f"❌ Force deletion failed (please check Drive permissions): {e}")
#     else:
#         print(f"✅ Confirmed path is physically cleared: {target_path}")

# print("\n>>> State refresh complete! The current Google Drive environment is confirmed to be completely clean.")
# print(">>> Please re-run [Cell 3] (Unified Environment Setup) immediately.")

# # Purpose: Completely clear project directories and cache backups using a dual approach of terminal rm -rf and Python's shutil.rmtree to resolve the "ghost folder" issue caused by cloud drive synchronization delays.
# # Upstream Code: User-provided native Python cleanup script.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-01 11:01 EDT.
# # Changed Lines:
# # * Added the !rm -rf terminal command as the first line of defense for cleanup.
# # * Optimized output logs to clearly distinguish between the terminal erasure and kernel verification phases.


>>> Initiating full-path forced cleanup sequence...
>>> [1/2] Forcing directory deletion via terminal commands...
>>> [2/2] Verifying kernel-state via Python...
✅ Confirmed path is physically cleared: /content/drive/MyDrive/Proteina-Complexa
✅ Confirmed path is physically cleared: /content/drive/MyDrive/Proteina_Subfolder_Backups

>>> State refresh complete! The current Google Drive environment is confirmed to be completely clean.
>>> Please re-run [Cell 3] (Unified Environment Setup) immediately.
